# Voice-Based Parkinson's Detection — Data, Pipeline and Results

**Sudipta Ghosh** · MSc thesis · generated from the live pipeline

This notebook walks the whole project in the order the work actually happens: what data we
have, what is inside the audio, how it is cleaned and preprocessed, what happens when we
train on mPower sustained vowels, how classical features compare with self-supervised
embeddings, and what survives when the locked model is applied to Spanish.

### The three research questions

| | |
|---|---|
| **RQ1** | How well do current single-modality voice-based PD methods generalise across datasets and languages? |
| **RQ2** | How can a voice method be designed to generalise across datasets/languages while keeping discrimination comparable to within-dataset performance? |
| **RQ3** | How should evaluation be designed to judge trustworthiness for clinical screening, and which measures support consistent cross-dataset/cross-language comparison? |

### How to read the numbers in this notebook

Two kinds appear, and they are labelled throughout:

- **COMPUTED HERE** — the cell you are looking at produced the number when the notebook ran.
- **CITED** — produced by a driver script in `scripts/`, loaded from `outputs/logs/*.json`.
  These need the full embedding caches and a specific cohort alignment, so they are not
  recomputed inline. The source file is always named.

Every number is **pilot-scale**: capped recordings per subject, one seed. See §13.

## Where these numbers come from

Nothing in this notebook is typed by hand. Every table and figure is computed when the
notebook runs, from result files that the scripts in `scripts/` produced from the audio.

```
scripts/*.py   ->   outputs/logs/*.json + *.csv   ->   this notebook
   compute              the results                   reads and explains
```

If a result file is missing, the cell says so rather than showing a stale number.


---
## §0 · Setup

In [ ]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
pd.set_option("display.width", 200, "display.max_columns", 50)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": .25, "axes.spines.top": False, "axes.spines.right": False})

from pdvoice.config import EMBEDDINGS, FIGURES, LOGS, SAMPLE_RATE
from pdvoice.data.ipvs import load_ipvs
from pdvoice.data.mdvr_kcl import load_mdvr_kcl
from pdvoice.data.mpower import load_mpower
from pdvoice.data.neurovoz import load_neurovoz

SEED = 42
WAVE, SPEC = "#3B6EA5", "#B5651D"

def cited(f):
    # Load a driver-script result JSON.
    return json.loads((LOGS / f).read_text())

print("sample rate :", SAMPLE_RATE, "Hz")
print("embeddings  :", EMBEDDINGS)
print("logs        :", LOGS)

# ---- artifact guard -------------------------------------------------------
# Every result table below is READ FROM A FILE produced by a driver script.
# Nothing is transcribed by hand. If a file is missing, the cell that needs it
# says which script to run rather than silently showing a stale number.
ARTIFACTS = {
    "label_sensitivity.json":                       "scripts/label_sensitivity.py",
    "classical_baseline__cap1.json":                "scripts/classical_baseline.py --cap 1",
    "fusion__xls-r-300m__L15.json":                 "scripts/fusion.py",
    "verify_ipvs_batch.json":                       "scripts/verify/verify_ipvs_batch.py",
    "scaleup_agematched__xls-r-300m__L15__cap10.json": "scripts/scaleup_agematched.py --cap 10 --seeds 42,43,44,45,46",
    "classifier_comparison.csv":                    "scripts/classifier_comparison.py",
    "silence_audit.csv":                            "scripts/audits/silence_audit.py",
    "neurovoz_taskcode_leak.json":                  "scripts/neurovoz_taskcode_leak.py",
    "literature_comparison.json":                   "scripts/literature_comparison.py",
    "embedding_probes.json":                        "scripts/embedding_probes.py",
    "objective_function.json":                      "scripts/objective_function.py",
    "encoder_layer_sweep.json":                     "scripts/encoder_layer_sweep.py",
    "sex_control.json":                             "scripts/sex_control.py",
    "vowel_phoneme_control.json":                   "scripts/vowel_phoneme_control.py",
    "amplitude_vs_harmonics__experimentA.csv":      "scripts/audits/amplitude_vs_harmonics.py",
    "amplitude_vs_harmonics__experimentB.csv":      "scripts/audits/amplitude_vs_harmonics.py",
    "group_spectra.csv":                            "scripts/audits/group_spectra.py",
}
missing = {f: cmd for f, cmd in ARTIFACTS.items() if not (LOGS / f).exists()}
print(f"\nartifacts present: {len(ARTIFACTS)-len(missing)}/{len(ARTIFACTS)}")
for f, cmd in missing.items():
    print(f"  MISSING {f:<48} -> run {cmd}")

def have(f):
    return (LOGS / f).exists()

---
# PART I — THE DATA

## §1 · Train and test data

The design is deliberately asymmetric, and this is the core of RQ1. One dataset trains the
model. The others are **never** trained on — the model is *locked* (weights, calibration map
and decision threshold all frozen) before they are touched.

That is the only way to measure generalisation honestly. Most published work in this field
reports within-dataset cross-validation and stops, which is why in-dataset 95–99% typically
falls to 70–85% on independent data.

In [ ]:
roles = pd.DataFrame([
    dict(corpus="mPower",   language="English", role="TRAIN",
         why="Largest by far; the only corpus big enough to train on. Crowd-sourced by phone."),
    dict(corpus="NeuroVoz", language="Spanish", role="TEST (locked)",
         why="Clinician-labelled. The primary cross-LANGUAGE test — §12."),
    dict(corpus="IPVS",     language="Italian", role="TEST (locked)",
         why="Clinician-labelled. Second language; elderly controls make it the hard case."),
    dict(corpus="MDVR-KCL", language="English", role="TEST (locked)",
         why="Clinician-labelled. Language held constant, so it isolates TASK + channel shift."),
])
roles

Holding one axis constant at a time is what makes the comparison interpretable:

- **mPower → NeuroVoz / IPVS** changes language *and* task *and* recording channel.
- **mPower → MDVR-KCL** keeps English, so whatever degrades is task and channel, not language.

That contrast turns out to matter a great deal — see §12.

---
## §2 · Type of audio

Two clinically distinct task families, and the distinction dominates every result in this
project.

**Sustained vowel** — the speaker holds a single vowel (`/a/`) as steadily as they can.
This isolates **phonation**: how stable the vocal folds are.

**Connected speech** — reading a passage, or spontaneous talking. This additionally exposes
**prosody**, **articulation** and **fluency**.

Parkinson's produces *hypokinetic dysarthria*, which degrades four separable systems. A
sustained vowel exposes only the first:

In [ ]:
pd.DataFrame([
    dict(system="Phonatory",       signature="jitter, shimmer, reduced HNR, breathiness, 4-6 Hz tremor",
         in_vowel="YES", in_speech="YES"),
    dict(system="Prosodic",        signature="monopitch, monoloudness, reduced stress",
         in_vowel="no",  in_speech="YES"),
    dict(system="Articulatory",    signature="imprecise consonants, reduced vowel space",
         in_vowel="no",  in_speech="YES"),
    dict(system="Fluency / timing", signature="inappropriate pauses, festination, short rushes",
         in_vowel="no",  in_speech="YES"),
])

**One of four versus four of four.** This is a stronger argument than "speech has more data",
and it predicts the result in §12: mPower is *vowel-only*, which structurally caps what a
model trained on it can learn.

It also explains why speech transfers across languages — prosodic timing and articulatory
imprecision are motor deficits, largely independent of which phonology is being degraded.

### What "vowel" and "speech" actually mean in each corpus

A caveat that belongs in the methods chapter: the experiments filter on `task_kind`
(`vowel` / `speech`) and **never** on `task_code`. The same word therefore denotes
different recordings in each corpus.

In [ ]:
mp, nv, ip, md_ = load_mpower(), load_neurovoz(), load_ipvs(), load_mdvr_kcl()
MANIFESTS = {"mpower": mp, "neurovoz": nv, "ipvs": ip, "mdvr_kcl": md_}

rows = []
for name, m in MANIFESTS.items():
    for (kind, code), g in m.groupby(["task_kind", "task_code"]):
        rows.append(dict(corpus=name, task_kind=kind, task_code=code,
                         recordings=len(g), subjects=g.subject_id.nunique(),
                         PD=int((g.label == 1).sum()), HC=int((g.label == 0).sum())))
tasks = pd.DataFrame(rows).sort_values(["corpus", "task_kind", "task_code"])
print("task codes per corpus (COMPUTED HERE)")
tasks.groupby(["corpus", "task_kind"]).agg(
    n_codes=("task_code", "nunique"), codes=("task_code", lambda s: ", ".join(sorted(s)[:6])),
    recordings=("recordings", "sum"))

So the vowel comparison is **cross-phoneme as well as cross-language**: we train on `/a/`
and test on `/a,e,i,o,u/` pooled. And "connected speech" ranges from a 1.75 s isolated
Spanish word to 141 s of English spontaneous dialogue. Restricting external vowel tests to
`/a/` only (`A*` in NeuroVoz, `VA` in IPVS) would isolate language from phoneme — an open
methodological item.

One more imbalance worth recording: NeuroVoz's **third** vowel repetition is almost
patient-only.

In [ ]:
v = tasks[(tasks.corpus == "neurovoz") & (tasks.task_kind == "vowel")][["task_code", "PD", "HC"]]
v = v.set_index("task_code").T
print("NeuroVoz vowel repetitions, PD vs HC recordings (COMPUTED HERE)")
v

`A3` is 29 PD / 3 HC, `I3` is 26 / 1, `O3` is 28 / 2 — while `U2` runs the other way at
5 / 23. Pooling all vowels means the two groups contribute **different task mixes** to their
subject-level averages: a composition difference layered on top of the disease difference.
NeuroVoz within-vowel is a modest 0.610, so little is being inflated, but it is a disclosed
limitation.

### How much of the diagnosis is in the file names alone?

The imbalance above is not cosmetic. If the *third repetition* survives mostly for patients,
then which task codes a speaker has is itself informative about the diagnosis — and that
information is in the file name, available to any model that is trained within this corpus,
with no audio involved.

This is worth measuring rather than worrying about. Three ways, all subject-level and all
using metadata only:

- **task-code profile** — score each recording by its code's PD share, average per speaker
- **recording count** — nothing but how many vowel files the speaker contributed
- **recording-level** — the code as the single feature, without aggregating to speakers

In [ ]:
if have("neurovoz_taskcode_leak.json"):
    lk = cited("neurovoz_taskcode_leak.json")
    per = pd.DataFrame(lk["per_code"]).set_index("task_code")
    display(per)

    probe = pd.DataFrame([
        dict(what="task-code profile (subject)", AUROC=lk["auroc_taskcode_profile"],
             ci=f"[{lk['ci_taskcode_profile'][0]:.3f}, {lk['ci_taskcode_profile'][1]:.3f}]"),
        dict(what="recording count alone (subject)", AUROC=lk["auroc_recording_count"],
             ci=f"[{lk['ci_recording_count'][0]:.3f}, {lk['ci_recording_count'][1]:.3f}]"),
        dict(what="task code alone (recording)", AUROC=lk["auroc_recording_level"], ci=""),
    ])
    print("PREDICTING PARKINSON'S FROM METADATA ONLY - no audio at any point")
    display(probe)
    rp = lk["recordings_per_speaker"]
    print(f"vowel recordings per speaker: HC {rp['HC']:.2f}  PD {rp['PD']:.2f}")
    print(f"third repetitions (A3/E3/I3/O3/U3): "
          f"{100 * lk['third_repetition_pd_share']:.0f}% patients")
else:
    print("run scripts/neurovoz_taskcode_leak.py")

In [ ]:
if have("neurovoz_taskcode_leak.json"):
    per = pd.DataFrame(cited("neurovoz_taskcode_leak.json")["per_code"]).set_index("task_code")
    fig, ax = plt.subplots(figsize=(8, 3.2))
    third = [c.endswith("3") for c in per.index]
    ax.bar(per.index, per.pd_share, color=[SPEC if t else WAVE for t in third])
    ax.axhline(.5, ls=":", c="k", lw=1)
    ax.set_ylabel("share of recordings that are PD"); ax.set_ylim(0, 1.05)
    ax.set_title("NeuroVoz vowel task codes: the third take is almost patients-only\n"
                 "orange = third repetition", fontweight="bold")
    for i, (n, s) in enumerate(zip(per.total, per.pd_share)):
        ax.text(i, s + .02, f"n={n}", ha="center", fontsize=6.5)
    fig.tight_layout(); plt.show()

**A speaker's task codes predict Parkinson's at 0.895, and the bare count of their recordings
at 0.790 — with no audio at all.** Controls average 8.6 vowel recordings, patients 10.7, and
the third repetitions are 93% patients. Whatever happened during collection, the third take
was largely not kept for the control group.

This is the same class of problem as the IPVS 44.1 kHz batch: **a corpus artefact that
correlates with the label.** A permutation test cannot see it, because it is a property of the
data rather than of the plumbing — permuting labels destroys the leak along with the signal.

Two consequences, and they are different in severity.

**For training within NeuroVoz on pooled vowels: serious.** Our own within-NeuroVoz vowel
figure (0.610, from `transfer_any.py --train neurovoz-vowel`) is measured on exactly this
design, and so is the published 0.646 it is compared against in §12b. Neither is confound-free,
and their agreement is not mutual validation. Any future within-corpus vowel model here must
balance by task code first.

**For transfer *into* NeuroVoz: not a problem, and this is checked rather than assumed.** A
locked mPower model never sees the file name, so it cannot use the code. The control two
sections above is the direct test: restricting the target to `/a/` only — which removes most
of the imbalance — moves transfer by −0.023, i.e. nothing. The transfer numbers in PART IV
stand.

### Control: is the cross-corpus vowel comparison really cross-*phoneme*?

The section above flagged that "vowel" is not one task. mPower records `/a/` only; NeuroVoz
and IPVS record several vowels per speaker. So a mPower->NeuroVoz drop could be a **phoneme**
mismatch rather than a language or disease effect.

The control restricts the target to `/a/` only and re-scores. It also equalises the number of
recordings per speaker - without that, "all vowels" averages several recordings per speaker
against 1 for `/a/`-only, and averaging more recordings reduces variance on its own. That second control is the one that matters: without it the
apparent effect is roughly twice as large.

In [ ]:
if have("vowel_phoneme_control.json"):
    vp = cited("vowel_phoneme_control.json")
    r = pd.DataFrame(vp["rows"])
    piv = r.pivot_table(index="subset", columns="target", values="auroc",
                        aggfunc=["mean", "std"]).round(3)
    print(f"subject-level AUROC over seeds {sorted(r.seed.unique())}")
    display(piv)
    print(f"mean effect of restricting the target to /a/ only: {vp['mean_delta']:+.4f} AUROC")
    print("(negative = matching the phoneme makes transfer WORSE, not better)")
else:
    print("run scripts/vowel_phoneme_control.py")

**The phoneme mismatch is not the explanation.** Restricting the target to `/a/` - the exact
phoneme the model was trained on - changes transfer by about **-0.02** AUROC on average, i.e.
slightly *worse*, and well inside the seed spread. If phoneme mismatch were driving the
collapse, removing it would have recovered performance.

Worth stating in the thesis because it is a cheap objection a reader will raise, and it is now
closed with a measurement rather than an argument.

---
## §3 · Sample size and audio format

Read from the audio file **headers**, not from the papers. Format (rate, bit depth,
channels) is a complete census of all four corpora.

In [ ]:
import soundfile as sf

rng = np.random.default_rng(SEED)
inv = []
for name, m in MANIFESTS.items():
    paths = list(m.path)
    probe = rng.choice(paths, 4000, replace=False) if len(paths) > 4000 else paths
    info = [sf.info(p) for p in probe]
    dur = np.array([i.frames / i.samplerate for i in info])
    sub = m.groupby("subject_id").label.first()
    inv.append(dict(
        corpus=name, language=m.language.mode()[0],
        recordings=len(m), subjects=m.subject_id.nunique(),
        subj_PD=int((sub == 1).sum()), subj_HC=int((sub == 0).sum()),
        subj_unlabelled=int(sub.isna().sum()),
        rec_per_subj=round(len(m) / m.subject_id.nunique(), 1),
        sample_rate=",".join(f"{k}" for k in sorted({i.samplerate for i in info})),
        bit_depth=",".join(sorted({i.subtype for i in info})),
        channels=",".join(str(c) for c in sorted({i.channels for i in info})),
        median_dur_s=round(float(np.median(dur)), 2),
        probed=len(probe)))
inventory = pd.DataFrame(inv)
print("DATASET INVENTORY (COMPUTED HERE)")
inventory

**Every one of the 61,492 files is single-channel mono.** Two format notes matter:

- **IPVS mixes sample rates** — 16 kHz and 44.1 kHz. That split is confounded with the label
  (§6).
- **MDVR-KCL is the only 24-bit corpus.** Practically negligible: 16-bit quantisation noise
  is already inaudible and everything is converted to float32 on load.

### Why 16 kHz is the right target, not a compromise

The **Nyquist limit** is half the sample rate — the highest frequency a recording can
physically represent. Audible speech energy sits below 8 kHz, so 16 kHz captures it all.
A higher rate adds no speech information and only costs compute. Every encoder in this field
is pretrained at 16 kHz, so we resample everything to 16 kHz on load.

Downsampling is *not* decimation: dropping every other sample folds high frequencies back
down as false low ones (**aliasing**). Correct downsampling low-passes first, which is why we
delegate to `librosa`.

### Sample size is wildly imbalanced, in two directions at once

In [ ]:
lab = mp[mp.label.isin([0, 1])]
per = lab.groupby(["label", "subject_id"]).size().groupby("label").describe()[["count","mean","50%","max"]]
print("mPower recordings per subject, by class (COMPUTED HERE)")
print(per.round(1).to_string())
print()
print(f"  recording level : {int((lab.label==1).sum()):>6,} PD  vs {int((lab.label==0).sum()):>6,} HC")
print(f"  subject level   : {lab[lab.label==1].subject_id.nunique():>6,} PD  vs "
      f"{lab[lab.label==0].subject_id.nunique():>6,} HC")

**This is a trap worth naming explicitly.** At *recording* level mPower looks PD-majority;
at *subject* level it is HC-majority roughly 4:1. Patients used the app far more (median 8
recordings, up to 355) than controls (median 3).

Three defences, all applied: **cap** recordings per subject, **class-weight** the loss, and
**evaluate at subject level** — a person's recording probabilities are averaged so one
speaker counts once, whether they contributed 1 recording or 355.

---
## §4 · Waveform — the time-domain view

The waveform plots sample amplitude against time. Values are floats in **[-1.0, 1.0]**.
It is the view for spotting timing, loudness, clipping, dropouts and noise.

In [ ]:
from pdvoice.preprocess import VOWEL_CORE_S, voice_activity_trim
from pdvoice.utils.audio import load_audio, rms_dbfs, stable_core, trim_silence
import librosa, librosa.display

def pick(m, task, label, seed=SEED):
    s = m[(m.task_kind == task) & (m.label == label)]
    return s.sample(1, random_state=seed).iloc[0]

def prep(row):
    w = load_audio(row.path)
    return (stable_core(trim_silence(w), seconds=VOWEL_CORE_S) if row.task_kind == "vowel"
            else voice_activity_trim(w)[: int(5 * SAMPLE_RATE)])

ex = {"HC · sustained vowel": pick(ip, "vowel", 0), "PD · sustained vowel": pick(ip, "vowel", 1),
      "HC · connected speech": pick(md_, "speech", 0), "PD · connected speech": pick(md_, "speech", 1)}
clips = {k: prep(r) for k, r in ex.items()}

fig, axes = plt.subplots(2, 2, figsize=(12, 5))
for ax, (k, w) in zip(axes.ravel(), clips.items()):
    librosa.display.waveshow(w, sr=SAMPLE_RATE, ax=ax, color=WAVE)
    ax.set_title(f"{k}   ({len(w)/SAMPLE_RATE:.2f} s, RMS {rms_dbfs(w):.1f} dBFS)", fontsize=9)
    ax.set_ylabel("amplitude")
fig.suptitle("Waveform — after preprocessing, 16 kHz", fontweight="bold")
fig.tight_layout(); plt.show()

Two things to read here, and the second is a warning.

**A sustained vowel is a steady band; connected speech is bursty** — visible onsets, silences
between words, varying amplitude. That structural difference is why the two tasks need
different preprocessing (§7).

**The RMS values in the titles differ substantially between recordings of the same task.**
That is microphone, distance and gain — not disease. Because each panel auto-scales its
y-axis, a quiet recording *looks* the same size as a loud one. Never read loudness off a
waveform plot. §7 shows how the encoder's normalisation removes exactly this nuisance.

---
## §5 · Spectrum, spectrogram and mel spectrogram

Four views of the *same data* — the waveform and the spectrum contain identical information,
one in time and one in frequency.

| View | Domain | Computed by | Shows |
|---|---|---|---|
| Waveform | time | raw samples | timing, loudness, noise |
| **Spectrum** | frequency | DFT/FFT on one window | harmonics at a frozen instant |
| **Spectrogram** | time–frequency | STFT (stacked DFTs) | how frequency evolves |
| **Mel spectrogram** | time–mel | STFT + mel filterbank | the same, warped to human hearing |

The DFT returns complex numbers; `np.abs()` keeps amplitude and discards the **phase**
spectrum, which is why a spectrogram cannot be inverted to audio without a vocoder. The
**mel** scale exists because human hearing resolves low frequencies far more finely than
high ones.

We use **speech-appropriate** analysis windows here — 1024-point FFT with a 160-sample hop
(64 ms / 10 ms) — rather than librosa's music-oriented defaults.

In [ ]:
N_FFT, HOP, N_MELS, FMAX = 1024, 160, 80, 8000

def four_views(w, sr, title):
    fig, ax = plt.subplots(1, 3, figsize=(13, 2.9))
    n = min(4096, len(w)); seg = w[max(0, len(w)//2 - n//2):][:n]
    dft = np.fft.rfft(seg * np.hanning(len(seg)))
    freq = librosa.fft_frequencies(sr=sr, n_fft=len(seg))
    ax[0].plot(freq, librosa.amplitude_to_db(np.abs(dft), ref=np.max), lw=.7, color=SPEC)
    ax[0].set_xscale("log"); ax[0].set_xlim(50, sr/2); ax[0].set_ylim(-80, 5)
    ax[0].set_title("spectrum (one window)", fontsize=9)
    ax[0].set_xlabel("frequency (Hz)"); ax[0].set_ylabel("dB")

    D = librosa.amplitude_to_db(np.abs(librosa.stft(w, n_fft=N_FFT, hop_length=HOP)), ref=np.max)
    librosa.display.specshow(D, sr=sr, hop_length=HOP, x_axis="time", y_axis="hz", ax=ax[1], cmap="magma")
    ax[1].set_ylim(0, FMAX); ax[1].set_title("spectrogram (linear Hz)", fontsize=9)

    M = librosa.feature.melspectrogram(y=w, sr=sr, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, fmax=FMAX)
    librosa.display.specshow(librosa.power_to_db(M, ref=np.max), sr=sr, hop_length=HOP,
                             x_axis="time", y_axis="mel", fmax=FMAX, ax=ax[2], cmap="magma")
    ax[2].set_title(f"mel spectrogram (n_mels={N_MELS})", fontsize=9)
    fig.suptitle(title, fontweight="bold", y=1.06)
    fig.tight_layout(); plt.show()

for k in ["HC · sustained vowel", "PD · sustained vowel"]:
    four_views(clips[k], SAMPLE_RATE, k)

In [ ]:
for k in ["HC · connected speech", "PD · connected speech"]:
    four_views(clips[k], SAMPLE_RATE, k)

In the **vowel** spectra, look for the evenly spaced peaks — the **harmonic comb**, whose
spacing is the fundamental frequency F0. The spectrogram shows them as flat horizontal
stripes, because a sustained vowel holds its pitch. Aperiodicity fills the gaps *between*
harmonics with noise, which is what **HNR** measures (§9).

In the **speech** panels the stripes break into segments as the articulators move, and
broadband vertical smears appear — fricatives (`/s/`, `/f/`, `/ʃ/`) with energy up to 8 kHz.
A sustained vowel has almost nothing up there. That difference matters again in §6.

> ⚠️ **Reproducibility trap.** A mel spectrogram is not a well-defined object unless you state
> the recipe. Two mel scales are in common use (`htk` and `slaney`), power and amplitude
> spectrograms both circulate, and "log-mel" sometimes means true decibels and sometimes a
> plain `log`. Undocumented variation here is one reason published voice-PD numbers are hard
> to compare.

> **These are four single recordings, chosen at random.** Do **not** read a disease signal off
> them. Showing a real PD/HC difference requires group averages over many speakers.

### The same comparison, averaged over many speakers

The four-views figures above each use **one randomly chosen recording per panel**. They show
what a spectrogram *is*; they cannot support a claim about PD versus HC. For that you need
group averages. You cannot average spectrograms across speakers — there is no time alignment
between different people — so the standard tool is the **Long-Term Average Spectrum**:
average power over time within a recording, then over speakers.

In [ ]:
from IPython.display import Image, display as _d
figdir = FIGURES / "writeup"
for f, cap in [("figD_group_spectra.png", "LTAS, PD vs HC, 150 speakers/group (scripts/audits/group_spectra.py)")]:
    if (figdir / f).exists():
        print(cap); _d(Image(filename=str(figdir / f)))
    else:
        print(f"MISSING {f} -> run scripts/audits/group_spectra.py")

if have("group_spectra.csv"):
    gs = pd.read_csv(LOGS / "group_spectra.csv")
    cols = [c for c in gs.columns if c.endswith("_dB")]
    print("\nfraction of frequency bins where |PD - HC| exceeds 2 dB:")
    print((gs[cols].abs() > 2).mean().round(3).to_string())

On the **raw** mPower cohort the two groups separate clearly. **Age-match the same corpus and
the separation largely collapses** — the difference falls to roughly what you would expect by
chance. That is the age confound appearing in the physics itself, before any model is fitted,
and it is an argument that lands without machine learning.

Three caveats. Adjacent FFT bins are correlated, so "fraction of bins differing" is
descriptive rather than a corrected significance test. **Sex is not controlled**, and the
region near 100 Hz is where F0 lives. And the age-matched panel showing almost nothing is
consistent with a modest ~0.65 AUROC: jitter, shimmer and tremor are *temporal* phenomena
that a time-averaged spectrum discards by construction.

---
# PART II — PROCESSING

## §6 · Cleaning process

Cleaning happens in two places: **cohort construction** (which recordings and which people
enter an experiment) and **quality control** (which files are usable at all).

```
load manifest        all recordings for a corpus, one row each
 → cap N per subject shuffle (seeded), keep first N — stops 355-recording speakers
                     dominating 1-recording speakers
 → quality control   drop duration < 1.0 s and unreadable files
 → label filter      keep only 0/1; self-report blanks dropped
 → age matching      5-year bins, 1:1 — the confound control (§8)
 → subject-disjoint  a speaker is never in both train and test
```

### The mPower cohort flow

In [ ]:
sub = mp.groupby("subject_id").label.first()
flow = pd.DataFrame([
    dict(step="survey respondents",                n_subjects=6805, PD="",    HC="",    note="Demographics Survey"),
    dict(step="completed the voice task",          n_subjects=5826, PD="",    HC="",    note="Voice Activity"),
    dict(step="labelled (professional-diagnosis)", n_subjects=4962, PD="970", HC="3,992", note="self-report"),
    dict(step="audio actually present on disk",    n_subjects=int((~sub.isna()).sum()),
         PD=f"{int((sub==1).sum()):,}", HC=f"{int((sub==0).sum()):,}",
         note="COMPUTED HERE — 153 speakers have a label but no file"),
    dict(step="after cap-3 + QC",                  n_subjects=4799, PD="890", HC="3,909", note="cited"),
    dict(step="after 5-year age matching",         n_subjects=1118, PD="559", HC="559",   note="cited; both mean age 56"),
])
flow

The fourth row was **missing from the original cohort diagram**. 153 speakers (73 patients,
80 controls) have a survey record and a label but no audio file. They were being removed
silently by the duration filter.

The cause was a truthy-string bug in the loader: when a file is missing, `str(NaN)` returns
the *non-empty* string `"nan"`, which passed the path guard. 7,410 of 65,022 manifest rows
(11.4%) pointed at nothing — and the drop was **not class-neutral** (15.3% of PD recordings
vs 5.7% of HC). Now fixed in `data/mpower.py`, which rejects placeholder paths and warns with
a count.

No published number is invalidated — QC was already deleting these as unreadable, which is
why the corrected counts reconcile with 890/3,909. But because **capping runs before QC**,
patients lost ~15% of their capped slots against ~5% for controls, which adds noise to the PD
class and pushes AUROC *toward* 0.5. Conservative, not inflated.

### Two disclosed data-quality limitations

**① IPVS has a patient-only recording batch.**

In [ ]:
s = ip.groupby("subject_id").agg(label=("label","first"), sr=("native_sr","max"))
print("IPVS subjects by label and native sample rate (COMPUTED HERE)")
print(s.groupby(["label","sr"]).size().rename("subjects").to_string())

from sklearn.metrics import roc_auc_score
print(f"\nAUROC of 'native_sr == 44100' alone, at subject level: "
      f"{roc_auc_score(s.label, (s.sr == 44100).astype(float)):.3f}")
try:
    b = cited("verify_ipvs_batch.json")
    print("\nCITED — scripts/verify/verify_ipvs_batch.py:")
    for k in ["baseline", "drop_44k_pd", "drop_44k_pd_and_young_hc"]:
        print(f"  {k:26s} AUROC = {b[k]['auroc']:.3f}  CI {b[k]['ci']}")
    print(f"  within-PD rate probe        AUROC = {b['within_pd_rate_probe']:.3f}")
except FileNotFoundError:
    print("\n(run scripts/verify/verify_ipvs_batch.py to populate)")

Native rate 44.1 kHz implies PD with perfect precision, and rate alone would score 0.679.
The artifact **is** encoded in the speech embeddings (within-patient probe 0.79) but **not**
the vowel ones (0.52) — consistent with fricative energy sitting near the 8 kHz anti-alias
rolloff, which only connected speech has.

**But it does not explain the IPVS ceiling.** Removing the entire batch leaves the AUROC at
1.000, and so does additionally removing the young controls. So this is a limitation to
disclose, not an explanation. *Honest limit on that test:* AUROC is saturated at 1.0, so it
shows the artifact is not **necessary** — it cannot show the artifact contributes nothing.

**② Quality control never opens the audio.** It reads the header for duration only, so a
uniformly quiet recording passes.

In [ ]:
if have("silence_audit.csv"):
    sa = pd.read_csv(LOGS / "silence_audit.csv")
    prevalence = sa.groupby("dataset").apply(lambda g: pd.Series({
        "measured": len(g),
        "all_zero": int(g.all_zero.fillna(False).sum()),
        "near_silent_lt_60dB": int((g.rms_dbfs < -60).sum()),
        "very_quiet_lt_50dB": int((g.rms_dbfs < -50).sum()),
    }), include_groups=False).reset_index()
    print("LOADED outputs/logs/silence_audit.csv\n")
    print(prevalence.to_string(index=False))
    tot, vq = prevalence.measured.sum(), prevalence.very_quiet_lt_50dB.sum()
    print(f"\n  {vq} of {tot:,} = {100*vq/tot:.2f}% very quiet")
    lab = sa[sa.label.isin([0, 1])]
    byc = lab.assign(vq=lab.rms_dbfs < -50).groupby(["dataset", "label"]).vq.mean() * 100
    print("\n  very-quiet rate (%) by dataset and class -- is it class-correlated?")
    print(byc.round(2).unstack().to_string())
else:
    print("run scripts/audits/silence_audit.py")

Low prevalence — but the mechanism is unguarded, and the consequence is counter-intuitive:
the encoder's normalisation **amplifies** a near-silent recording to full scale (+55 to
+70 dB), so it encodes the character of the *room* at the same level as real speech. Two
digitally silent clips also produce byte-identical embeddings, so they would form a perfect
cluster if they were ever class-correlated.

RMS level also cannot detect *audible but not voice* — a −30 dBFS recording of traffic passes
every check. Catching that needs a voicing test (F0 detected, or a VAD speech-fraction floor).

The guard is written but deliberately **opt-in and off by default**: enabling it silently
would change every existing cohort and hence every reported number.

### Supervisor ask 1 - is a quiet recording an uninformative recording?

The instruction was: **do not discard audio by amplitude.** Loudness is a property of the
*recording chain* - how far the phone was from the mouth, what gain the OS applied - not of
the larynx. Throwing away quiet files would discard signal and, worse, discard it
*non-randomly*.

Two experiments, and they answer different questions.

**Experiment A is the causal test.** Take one file and attenuate it by 0, -20 and -40 dB.
Nothing about the speaker changed, so any measure that moves is measuring the recording
level rather than the voice.

**Experiment B is observational.** Compare *real* quiet files against normal ones. Here a
difference is expected - but it cannot be attributed to level alone, because a quiet file is
also likely to be a badly recorded one.

In [ ]:
A = "amplitude_vs_harmonics__experimentA.csv"
MEAS = ["jitter", "shimmer", "hnr", "f0", "voiced_frac", "n_harmonic_peaks"]

if have(A):
    a = pd.read_csv(LOGS / A)
    # NOTE: never display the `path` column - it identifies participant recordings.
    g = a.groupby(["atten_db", "stage"])[["rms_dbfs"] + MEAS].mean().round(4)
    print("EXPERIMENT A - the same files, attenuated. Does the voice measurement move?")
    display(g)

    asis = a[a.stage == "as-is"].groupby("atten_db")[MEAS].mean()
    drift = (asis.loc[-40] - asis.loc[0]).abs()
    print("absolute change in each measure from 0 dB to -40 dB attenuation (as-is):")
    print(drift.round(5).to_string())
    print(f"\nlargest drift across all {len(MEAS)} measures: {drift.max():.2e}")
else:
    print("run scripts/audits/amplitude_vs_harmonics.py")

In [ ]:
B = "amplitude_vs_harmonics__experimentB.csv"
if have(B):
    b = pd.read_csv(LOGS / B)
    print("EXPERIMENT B - real quiet files vs normal ones (all normalised to -23 dBFS)")
    display(b.groupby("group")[MEAS].agg(["mean", "count"]).round(3))

    # How many of each group are unmeasurable at all (Praat returns NaN, no voicing)?
    nan_rate = b.assign(unvoiced=b.jitter.isna()).groupby("group").unvoiced.mean()
    print("fraction with NO measurable f0 (Praat returns NaN):")
    print((nan_rate * 100).round(1).astype(str).add(" %").to_string())
else:
    print("run scripts/audits/amplitude_vs_harmonics.py")

**Experiment A settles the principle.** Attenuating a file by 40 dB - a factor of 100 in
amplitude - leaves jitter, shimmer, HNR, F0 and the harmonic peak count numerically
unchanged. The harmonic structure is *scale-invariant*; only the RMS moves, exactly as it
should. So amplitude alone is not a reason to reject a recording, and the supervisor's
instruction is correct on the physics.

**Experiment B is where honesty is required, because it looks like the opposite.** Real quiet
files *are* worse: HNR around 2 dB against 17, jitter roughly 7x higher, and a small fraction
of frames voiced. But read the NaN row first - a large share of the quiet files have **no
measurable F0 at all**. Those are not quiet recordings of speech; many are recordings of a
room. The level is a *symptom* of a failed capture, not the cause of the degradation.

That distinction decides the pipeline design, and it is why the rule became **normalise, then
gate on voicing**:

- **amplitude** is not a valid rejection criterion (Experiment A)
- **absence of voicing** is one (Experiment B)
- normalisation to -23 dBFS is applied to *everything*, so the encoder sees a consistent level
  either way

Note also that `n_harmonic_peaks` is nearly identical in both groups - a peak-counting measure
survives what the periodicity measures do not.

---
## §7 · Preprocessing — every stage, input and output

Traced on real recordings. Nothing here is re-implemented; each stage calls the same code the
experiments call.

In [ ]:
import torch
from pdvoice.features.ssl import load_encoder, pool, _embed_wave
from pdvoice.preprocess import (CHUNK_S, TARGET_DBFS, chunks,
                                preprocess_clip)
from pdvoice.utils.audio import encoder_applied_gain_db, normalise_amplitude

model, extractor, device = load_encoder("facebook/wav2vec2-xls-r-300m")
print("encoder :", model.config.num_hidden_layers, "layers, hidden", model.config.hidden_size)
print("device  :", device)
print("params  :", f"{sum(p.numel() for p in model.parameters()):,} (FROZEN — never trained)")

In [ ]:
def trace(row):
    info = sf.info(row.path); out = []
    def add(step, i, o, extra=""):
        out.append(dict(step=step, input=i, output=o, note=extra))
    add("0 raw file", f"{info.samplerate} Hz {info.subtype}",
        f"{info.frames:,} samples · {info.frames/info.samplerate:.2f} s", f"{info.channels} channel")
    w = load_audio(row.path)
    add("1 load_audio", f"{info.samplerate} Hz int", f"{len(w):,} samples · float32 16 kHz",
        f"RMS {rms_dbfs(w):.1f} dBFS · values in [-1,1]")
    if row.task_kind == "vowel":
        t = trim_silence(w); add("2 trim_silence", f"{len(w):,}", f"{len(t):,} samples · {len(t)/SAMPLE_RATE:.2f} s", "top_db=30")
        c = stable_core(t, seconds=VOWEL_CORE_S); add("3 stable_core", f"{len(t):,}", f"{len(c):,} samples · {len(c)/SAMPLE_RATE:.2f} s", "central core")
        cl = [c]
    else:
        t = voice_activity_trim(w); add("2 silero VAD", f"{len(w):,} · {len(w)/SAMPLE_RATE:.1f} s",
            f"{len(t):,} · {len(t)/SAMPLE_RATE:.1f} s", f"{100*(1-len(t)/len(w)):.0f}% removed as non-speech")
        cl = list(chunks(t, seconds=CHUNK_S)); c = cl[0]
        add("3 chunks", f"{len(t):,}", f"{len(cl)} × {len(c):,} samples", "5 s, 50% overlap; embeddings averaged")
    # Supervisor ask 2: normalisation is OUR named stage, not a library side effect.
    c_raw, before = c, rms_dbfs(c)
    c = normalise_amplitude(c, target_dbfs=TARGET_DBFS)
    add("4 normalise_amplitude", f"RMS {before:.1f} dBFS",
        f"RMS {rms_dbfs(c):.1f} dBFS",
        f"EBU R128 target {TARGET_DBFS:.0f} dBFS · gain {rms_dbfs(c) - before:+.1f} dB")
    iv = extractor(c, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    a = iv.input_values.numpy()[0]
    add("5 feature extractor", f"RMS {rms_dbfs(c):.1f} dBFS", f"RMS {rms_dbfs(a):.1f} dBFS",
        f"zero-mean/unit-var; would have applied {encoder_applied_gain_db(c_raw):+.0f} dB implicitly · NO mel spectrogram")
    with torch.no_grad():
        hs = model(iv.input_values.to(device), output_hidden_states=True).hidden_states[15]
    hs = hs.squeeze(0).float().cpu().numpy()
    add("6 XLS-R layer 15", f"waveform ({len(c):,},)", f"hidden states {hs.shape}",
        f"~{1000*len(c)/SAMPLE_RATE/hs.shape[0]:.1f} ms per frame · FROZEN")
    v = pool(hs, "mean_std")
    add("7 mean+std pooling", f"(T={hs.shape[0]}, D={hs.shape[1]})", f"{v.shape} = 2048-dim",
        "mean = timbre · std = temporal variability")
    add("8 float16 cache", f"{v.nbytes} bytes float32", f"{v.astype(np.float16).nbytes} bytes float16", ".npz on disk")
    return pd.DataFrame(out)

print("=== mPower · sustained vowel · PD ===")
display(trace(pick(mp, "vowel", 1)))
print("=== MDVR-KCL · connected speech · PD ===")
display(trace(pick(md_, "speech", 1)))

### The five things to take from that table

**Stage 3 — the two tasks diverge.** A sustained vowel gives one 2 s central core (onset and
offset are the unstable parts of a phonation). Connected speech is cut into 5 s chunks whose
embeddings are averaged, so a 141 s recording and a 3 s one both become one vector.

**Stage 4 — normalisation, and why it is a stage at all (supervisor ask 2).** Every clip is
brought to **−23 dBFS**, the EBU R128 reference level, before it reaches the encoder. The
`gain` column shows how much each individual file needed, which is a direct measure of how
inconsistent the recording levels were.

This looks redundant, and in effect it is: the feature extractor at stage 5 zero-means and
unit-variances the waveform anyway, so for the XLS-R path the two together are equivalent to
the extractor alone. It is here for three reasons that outlast this model. Leaving it implicit
meant the pipeline had a step no reader could see. Stage 5 now also reports the gain the extractor
*would have* applied to the un-normalised clip — a real property of the pipeline that was
previously invisible, and one that varies by tens of dB between corpora. And any
model whose front end does *not* normalise would have inherited raw recording levels without
warning.

**Stage 5 — no mel spectrogram is computed anywhere.** XLS-R is a wav2vec2-family model that
consumes the raw waveform. Mel spectrograms belong to a different model family (Whisper).
Getting this wrong in a methods chapter is a common error.

**Stage 6 — the frame rate is 20 ms.** A 7-layer strided convolutional front-end learns its
own filterbank and emits one frame every ~20 ms with a ~25 ms receptive field. A consequence
worth stating: **jitter** is cycle-to-cycle variation of a glottal period roughly 10 ms long
— *below* the frame rate — so XLS-R cannot represent jitter as a period measurement, only its
spectral consequence. Vocal **tremor** at 4–6 Hz sits comfortably above it. This is a
principled reason the classical features and the embeddings measure partly different physics,
and it predicts §11.

**Stages 7–8 — a fixed 2048-dim vector at 4,096 bytes.** Cached once; every later experiment
trains on the cache in seconds instead of re-running a 315M-parameter model.

### Normalization, summarised

| Where | What |
|---|---|
| Waveform → encoder | zero-mean, unit-variance per recording (`Wav2Vec2FeatureExtractor`) |
| Classical features | median-impute then z-score |
| Inside the head | `BatchNorm1d(256)` |
| Probabilities | isotonic calibration, fitted on validation then **locked** |

---
# PART III — TRAINING ON mPOWER SUSTAINED VOWEL

## §8 · First model: mPower, sustained vowel

### The architecture, and what "frozen" means

Two models stacked, and only the small one learns:

```
audio → [FROZEN XLS-R-300M, 315,438,720 weights] → 2048 numbers → cached
                                                        ↓  + PD/HC label
                              [MLP HEAD, 525,313 weights, TRAINED] → p(PD)
```

**Frozen** and **locked** are different words for different guarantees:

| Term | Applies to | Means | Guards against |
|---|---|---|---|
| **Frozen** | the encoder | weights **never** updated by anyone, ever | overfitting 315M params to ~900 patients |
| **Locked** | head, calibrator, threshold | trained, then sealed **before** test/external data | optimistic bias from tuning on what you report |

The encoder was pretrained by Meta on 436,000 hours of **unlabelled** speech in 128
languages. It has never seen a diagnosis. The label enters at exactly one line —
`loss = loss_fn(model(xt), yt)` in `models/head.py`.

In [ ]:
from pdvoice.models.head import build_head
h = build_head(2048)
print(h)
enc = sum(p.numel() for p in model.parameters()); hd = sum(p.numel() for p in h.parameters())
print(f"\nencoder (frozen)  {enc:,}")
print(f"head    (trained) {hd:,}")
print(f"trained fraction  {100*hd/(enc+hd):.3f}%")

The hidden layer activation is **ReLU** — not softmax, which is an output activation for
multi-class problems. The output is a single **logit**; the sigmoid is applied inside
`BCEWithLogitsLoss` during training (numerically stable) and inside the calibrator at
inference. For two classes, sigmoid on one logit is mathematically identical to softmax over
two.

ReLU is also the only reason this is not a linear classifier: without it, `Linear(2048→256)`
followed by `Linear(256→1)` collapses algebraically into a single `Linear(2048→1)`, i.e.
plain logistic regression.

### Which encoder, and which layer? — the Layer-1 sweep

In [ ]:
rows = []
for f in sorted(LOGS.glob("layer1_mpower__*.json")):
    d = cited(f.name)
    p = f.stem.split("__")
    rows.append(dict(encoder=p[1], layer=p[3].replace("LNone", "last").replace("L", ""),
                     subject_AUROC=round(d.get("subject_auroc", d.get("auroc", float("nan"))), 3)))
sweep = pd.DataFrame(rows).pivot_table(index="encoder", columns="layer", values="subject_AUROC")
print("CITED — scripts/layer1_mpower.py · within-mPower 5-fold, cap-3, subject-level AUROC")
sweep

**XLS-R-300M wins**, and the layer result is more interesting than the encoder result:

- The English-only deep models (WavLM, HuBERT) do **better at a middle layer** than at the last.
- XLS-R improves with depth — **last layer is best within mPower (0.839)**.
- But **layer 15 transfers better** than the last layer. The within-domain-optimal layer is
  *not* the transfer-optimal layer: the last layer's mPower edge was partly domain-specific.

L15 is therefore used for all transfer work.

### ⚠️ The finding that reframes everything: it is largely an AGE confound

### The Layer-1 sweep was not a fair fight - the equal-budget re-run

The table above searched **four layers for XLS-R and one for wav2vec2-base**, and that one was
the last layer. So "XLS-R wins" was partly a statement about how much searching each encoder
received. Two further problems: it ran on the **confounded** cohort, so it partly measures
which encoder best encodes age; and `output_hidden_states=True` returns every layer from a
single forward pass, so the unequal budget bought nothing.

`encoder_layer_sweep.py` fixes all three: **every layer of every encoder**, on a fixed
**age-matched** cohort, one recording per speaker, identical audio for all five. It also tests
the SUPERB-style learned softmax weighting over layers.

In [ ]:
if have("encoder_layer_sweep.json"):
    es = cited("encoder_layer_sweep.json")
    best = pd.DataFrame(es["best"]).set_index("model")
    best["95% CI"] = [f"[{lo:.3f}, {hi:.3f}]" for lo, hi in zip(best.ci_lo, best.ci_hi)]
    show = best[["n_layers", "best_layer", "best_layer_auroc", "95% CI",
                 "last_layer_auroc", "gain_over_last",
                 "learned_pool_auroc", "learned_vs_best"]]
    print("EQUAL BUDGET - every layer searched, age-matched cohort, identical audio")
    display(show.round(4))
    print(f"mean gain from not using the last layer: {best.gain_over_last.mean():+.4f}")
    print(f"spread of best-layer AUROC across the {len(best)} encoders: "
          f"{best.best_layer_auroc.min():.3f} - {best.best_layer_auroc.max():.3f}")
else:
    print("run scripts/encoder_layer_sweep.py")

In [ ]:
PL = "encoder_layer_sweep__per_layer.csv"
if have(PL):
    pl = pd.read_csv(LOGS / PL)
    fig, ax = plt.subplots(figsize=(8, 3.6))
    for m, g in pl.groupby("model"):
        # x as a fraction of depth, so encoders with 13 and 25 layers overlay sensibly
        ax.plot(g.layer / g.layer.max(), g.auroc, marker="o", ms=2.5, lw=1.2, label=m)
    ax.axhline(.5, ls=":", c="k", lw=1)
    ax.set_xlabel("relative depth  (0 = input, 1 = last layer)")
    ax.set_ylabel("subject AUROC")
    ax.set_title("Every encoder peaks EARLY; the last layer is the worst choice for all five",
                 fontweight="bold")
    ax.legend(fontsize=7, ncol=3)
    fig.tight_layout()
    plt.show()
else:
    print(f"run scripts/encoder_layer_sweep.py  (missing {PL})")

**Two results, and the second undercuts a claim this project used to make.**

**The last layer is the worst layer for all five encoders**, with a mean gain of roughly +0.06
AUROC from simply choosing a better one. That is a larger effect than any encoder choice, and
it is free. The mechanism is standard in the SSL-probing literature: the final layers
specialise toward the pretraining objective, and for a discriminative health task the useful
representation sits earlier.

**No encoder is distinguishable from another once the budget is equal.** The best-layer AUROCs
fall in a narrow band and the confidence intervals overlap heavily. The earlier "XLS-R-300M
wins" was a search-effort artefact. Note carefully what this does *not* say: it does not say
the unequal search *changed the winner* - the significance tests do not support that. It says
the comparison cannot separate the encoders at all.

The SUPERB-style learned layer weighting does **not** beat picking the single best layer for
most of the encoders, which is a mildly useful negative result of its own.

In [ ]:
cf = pd.read_csv(LOGS / "classical_features_labeled__cap1.csv")
print("mPower age by class (COMPUTED HERE)")
print(cf.groupby("label").age.describe()[["count", "mean", "50%", "min", "max"]].round(1).to_string())
gap = cf[cf.label==1].age.mean() - cf[cf.label==0].age.mean()
print(f"\n  MEAN AGE GAP: {gap:.1f} years (PD older)")

fig, ax = plt.subplots(figsize=(7, 2.6))
for l, c, n in [(0, WAVE, "HC"), (1, SPEC, "PD")]:
    ax.hist(cf[cf.label == l].age.dropna(), bins=40, alpha=.6, color=c, label=f"{n} (n={int((cf.label==l).sum())})")
ax.set_xlabel("age (years)"); ax.set_ylabel("subjects")
ax.set_title("mPower age distributions — the confound", fontweight="bold")
ax.legend(); fig.tight_layout(); plt.show()

mPower controls are roughly **27 years younger** than the patients (subject-level: HC 32.6
vs PD 60.0, computed above). The app was adopted by young healthy volunteers and older
patients. A model can therefore score well by reading **age** rather than disease — and the
embeddings do encode age: on mPower controls a young-vs-elderly probe reaches AUROC **0.761**,
and continuous age gives **R² 0.177** (§10). A much higher figure is sometimes quoted for this
probe; that comes from IPVS, a small anechoic corpus with an extreme age contrast, and does
not hold on mPower's phone audio.

We can test whether labels or age is the bottleneck — five seeds each, because the confounded
within score alone varies by ±0.05 between seeds:


In [ ]:
LABELS = {"baseline": "Baseline (all labelled)", "confident": "Confident labels only",
          "age50": "Age cut (>= 50)", "agematched": "Age-matched (5-yr bins)"}
KEYS = [("within", "within"), ("ipvs", "transfer_ipvs"),
        ("neurovoz", "transfer_neurovoz"), ("mdvr_kcl", "transfer_mdvr_kcl")]
if have("label_sensitivity.json"):
    ls = cited("label_sensitivity.json")
    rows, base = [], None
    for k, runs in ls["runs"].items():
        v = {c: np.array([r[j] for r in runs], float) for c, j in KEYS}
        if base is None:
            base = {c: v[c].mean() for c in v}
        rows.append(dict(
            definition=LABELS.get(k, k), n_PD=runs[0]["n_pd"], n_HC=runs[0]["n_hc"],
            **{c: f"{v[c].mean():.3f}+/-{v[c].std(ddof=1):.3f}" for c, _ in KEYS},
            d_within=round(v["within"].mean() - base["within"], 3),
            d_transfer=round(float(np.mean([v[c].mean() - base[c] for c, _ in KEYS[1:]])), 3)))
    print(f"LOADED outputs/logs/label_sensitivity.json  ·  cap-{ls['cap']}, "
          f"{len(ls['seeds'])} seeds, within = layer {ls['within_layer']}, "
          f"transfer = L{ls['transfer_layer']}")
    print("d_within / d_transfer are deltas against the baseline row.")
    print(f"age gap: PD {ls['age_pd']} vs HC {ls['age_hc']}"
          f" = {ls.get('age_gap'):.1f} yr, per {ls.get('age_level', '?')}"
          f" -- agrees with the value computed above.\n")
    display(pd.DataFrame(rows))
else:
    print("run scripts/label_sensitivity.py")


**Read the rows against each other.** Every value is the mean over five seeds; `±` is the
SD across them, and the last two columns are deltas against the baseline row.

- Cleaning the **labels** moves the within score **+0.011** and mean transfer **+0.010**.
  Both sit well inside the seed noise, and the Italian inversion survives untouched. So
  self-report label noise is *not* the bottleneck.
- **Age-matching collapses the within score (− 0.185)** and simultaneously **lifts transfer
  (+0.124)**, turning Italian from 0.404 ± 0.106 into 0.624 ± 0.086.

That *opposition* is the whole diagnosis, and it is worth stating as a rule: a confound
produces exactly this pair — remove it and apparent within-dataset performance falls while
genuine transfer rises. A real modelling improvement would raise both. Label noise, tested
identically on the same seeds, moves neither. The intermediate row (age ≥ 50 cut) lands where
you would expect: a cruder control giving a smaller effect in the same direction
(− 0.168 / +0.054).

Roughly **0.19 of the confounded 0.80** was the model reading age rather than disease.

Two things this does **not** say. It does not say age accounts for *all* of the headline
number — what survives matching is a real, modest signal (§12). And it does not licence
single-seed comparisons: the confounded within score is **0.801 ± 0.050**, a spread wide
enough that two runs of the *same* configuration can differ by more than the label effect
being measured. That is why everything in this section is multi-seed.

### The honest age-matched model

The canonical version uses proper 5-year-bin frequency matching. Let us rebuild that cohort here:


In [ ]:
def age_match_ids(df, seed=SEED, bin_width=5):
    # Frequency-match PD & HC age per bin: keep min(nPD, nHC) of each.
    s = df.dropna(subset=["age"]).drop_duplicates("subject_id")[["subject_id", "label", "age"]]
    s = s.assign(bin=(s.age // bin_width * bin_width).astype(int))
    rng = np.random.default_rng(seed); keep = []
    for _, g in s.groupby("bin"):
        p_, h_ = g[g.label == 1].subject_id.to_numpy(), g[g.label == 0].subject_id.to_numpy()
        k = min(len(p_), len(h_))
        if k:
            keep += rng.choice(p_, k, replace=False).tolist() + rng.choice(h_, k, replace=False).tolist()
    return set(keep)

keep = age_match_ids(cf)
am = cf[cf.subject_id.isin(keep)]
print("COMPUTED HERE — age-matched cohort")
print(f"  before : {len(cf):,} subjects | PD {int((cf.label==1).sum()):,} (age {cf[cf.label==1].age.mean():.1f}) "
      f"| HC {int((cf.label==0).sum()):,} (age {cf[cf.label==0].age.mean():.1f})")
print(f"  after  : {len(am):,} subjects | PD {int((am.label==1).sum()):,} (age {am[am.label==1].age.mean():.1f}) "
      f"| HC {int((am.label==0).sum()):,} (age {am[am.label==0].age.mean():.1f})")
print(f"  residual age gap: {am[am.label==1].age.mean() - am[am.label==0].age.mean():+.2f} years")

In [ ]:
# The pilot single-seed model has been superseded by the 5-seed scale-up, and the
# script that produced it (agematched_eval.py) was removed as a duplicate of
# scaleup_agematched.py. Report the scaled result instead.
SC = "scaleup_agematched__xls-r-300m__L15__cap10.json"
if have(SC):
    sc = cited(SC)
    print(f"LOADED {SC}")
    print(f"  seeds {sc['seeds']} | cohort {sc['runs'][0]['n_subjects']:,} speakers "
          f"({sc['runs'][0]['n_recordings']:,} recordings), cap-{sc['cap']}")
    for key, lbl in [(None, "within mPower"), ("neurovoz", "-> NeuroVoz (es)"),
                     ("ipvs", "-> IPVS (it)"), ("mdvr_kcl", "-> MDVR-KCL (en)")]:
        v = np.array([(r["within"] if key is None else r["transfer"][key])["auroc"]
                      for r in sc["runs"]], float)
        e = np.array([(r["within"] if key is None else r["transfer"][key])["ece"]
                      for r in sc["runs"]], float)
        print(f"  {lbl:<18} AUROC {v.mean():.3f} +/- {v.std(ddof=1):.3f}   "
              f"ECE {e.mean():.3f} +/- {e.std(ddof=1):.3f}")
else:
    print("run scripts/scaleup_agematched.py --cap 10 --seeds 42,43,44,45,46")


**Headline: within-mPower 0.621 [0.59, 0.65]**, down from a confounded 0.803 — and the
transfer gap essentially vanishes (§12). The saved model is in
`outputs/models/agematched__xls-r-300m__L15__{head.pt,calibrator.joblib}`.

This is the thesis spine: *reported voice-PD accuracy is inflated by demographic confounds;
controlled for it, performance is modest but genuinely generalisable.*

### The decisive control: how well does AGE ALONE do, with no audio at all?

Borrowed from the Bridge2AI VoiceFM paper, which includes a demographics-only baseline
"to rule out the possibility that AUROC is driven by demographic confounding rather than
acoustic disease information."

This is the single most informative row in the project. If a model cannot beat a logistic
regression on the participant's age with **no microphone involved**, it has not
demonstrated a voice biomarker.

In [ ]:
if have("classifier_comparison.csv"):
    cc = pd.read_csv(LOGS / "classifier_comparison.csv")
    for cohort in ("confounded", "age-matched"):
        piv = (cc[cc.cohort == cohort]
               .pivot(index="classifier", columns="features", values="AUROC"))
        print(f"=== {cohort.upper()} cohort — subject-level AUROC ===")
        display(piv.round(3))
    print("LOADED outputs/logs/classifier_comparison.csv")
else:
    print("run scripts/classifier_comparison.py")

In [ ]:
if have("classifier_comparison.csv"):
    cc = pd.read_csv(LOGS / "classifier_comparison.csv")
    best = (cc.groupby(["cohort", "features"]).AUROC.max().unstack())
# Order the bars by how much the model is allowed to see. Names are matched by
# PREFIX, not by the exact string: the classical set has grown from 17 features to
# 42, and an exact-match list silently dropped that bar from this chart instead of
# raising -- the figure simply showed two bars where it should have shown three.
    order = [c for key in ("age only", "classical", "SSL embeddings")
             for c in best.columns if c.startswith(key)]
    missing = [c for c in best.columns if c not in order]
    assert not missing, f"a feature set is not being plotted: {missing}"
    best = best[order]

    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    x = np.arange(len(best.columns)); w = 0.38
    for i, (cohort, colour) in enumerate([("confounded", SPEC), ("age-matched", WAVE)]):
        ax.bar(x + (i - .5) * w, best.loc[cohort], w, color=colour, label=cohort)
    ax.axhline(.5, ls=":", c="k", lw=1)
    ax.set_xticks(x); ax.set_xticklabels(best.columns)
    ax.set_ylabel("best subject AUROC"); ax.set_ylim(.4, 1.0); ax.legend(fontsize=8)
    ax.set_title("Age alone beats every audio model — until you age-match",
                 fontweight="bold")
    fig.tight_layout(); plt.show()
    display(best.round(3))

**Read the orange bars.** On the confounded cohort, a single feature — the participant's
age, no audio — outscores 2048-dimensional XLS-R embeddings and the 42 classical features.
Every published mPower number obtained without demographic control is competing against a
baseline that needs no microphone.

**Read the blue bars.** After age-matching, age collapses to chance, which independently
**verifies that the matching worked**, and the audio features retain a modest but real
signal. That pair of facts is the thesis in one figure.

### Age is not the only confound - sex is a second, independent one

The age result above is the headline, but it is not the whole demographic story. In mPower the
patient group is substantially more female than the control group, and **the embeddings encode
sex almost perfectly** (see the probe section later). So a model has unrestricted access to
sex whether or not we intend it.

`sex_control.py` asks whether matching on sex *as well* costs anything. Because it matches
within age bins, it uses the same speakers - the question is whether the extra constraint is
free.

In [ ]:
if have("sex_control.json"):
    sx = pd.DataFrame(cited("sex_control.json")).set_index("cohort")
    sx["SSL 95% CI"] = [f"[{lo:.3f}, {hi:.3f}]" for lo, hi in sx["ci"]]
    show = sx[["n", "pct_female_hc", "pct_female_pd", "auroc_age", "auroc_sex",
               "auroc_age_sex", "auroc_ssl", "SSL 95% CI",
               "ssl_within_male", "ssl_within_female"]]
    display(show.round(4))
    gap = (sx.pct_female_pd - sx.pct_female_hc).round(1)
    print("PD-minus-HC percentage-point gap in % female, by cohort:")
    print(gap.astype(str).add(" pp").to_string())
else:
    print("run scripts/sex_control.py")

**Read the `pct_female` columns first.** On the raw cohort the patient group is far more female
than the control group - a gap of roughly 16 percentage points. **Age-matching barely touches
it**, because age and sex are not the same variable; correcting one does not correct the other.
Sex alone predicts the label at about 0.58, which is small but real.

**Matching on age *and* sex is essentially free.** It uses the same number of speakers and the
SSL AUROC does not fall - if anything it is marginally higher than the age-only-matched cohort,
and the CIs overlap. Since it costs nothing and closes a documented confound, **age+sex
matching should be the default**, and the thesis should report it as such.

One honest caution about the last two columns. The within-sex scores are computed on much
smaller subgroups, so the female column in particular is noisy; the apparent male/female
difference on the age+sex-matched cohort is not something to interpret.

---
## §9 · Classical features — the interpretable baseline

42 hand-engineered coefficients, each mapping to a speech measure a clinician recognises —
unlike a dimension of a 2048-vector.

| Feature | Measures | Elevated / reduced in PD |
|---|---|---|
| **jitter** (local) | cycle-to-cycle variation in *period* | elevated — unstable pitch |
| **shimmer** (local) | cycle-to-cycle variation in *amplitude* | elevated — unstable loudness |
| **HNR** | harmonic-to-noise ratio | reduced — breathy, noisy voice |
| **F0 mean** | fundamental frequency (pitch) | shifted |
| **MFCC 0–12** | spectral envelope (vocal-tract shape) | altered |

Computed with Praat via `parselmouth` at the file's native rate (period measurement needs the
resolution), MFCCs at 16 kHz. These are designed for **sustained phonation** and are not
reliable on connected speech — which is why the classical arm exists only for the vowel task.

In [ ]:
FEATS = ["jitter_local", "shimmer_local", "hnr", "f0_mean"] + [f"mfcc_{i}" for i in range(13)]
print(f"COMPUTED HERE — {len(FEATS)} features over {len(cf):,} subjects (cap-1)")
display(cf[FEATS[:4]].describe().round(4))

fig, axes = plt.subplots(1, 4, figsize=(13, 2.6))
for ax, f in zip(axes, ["jitter_local", "shimmer_local", "hnr", "f0_mean"]):
    for l, c, n in [(0, WAVE, "HC"), (1, SPEC, "PD")]:
        d = cf[cf.label == l][f].dropna()
        d = d[(d > d.quantile(.01)) & (d < d.quantile(.99))]
        ax.hist(d, bins=45, alpha=.6, color=c, label=n, density=True)
    ax.set_title(f, fontsize=9); ax.set_yticks([])
axes[0].legend(fontsize=8)
fig.suptitle("Classical voice features, PD vs HC (confounded cohort)", fontweight="bold")
fig.tight_layout(); plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from pdvoice.eval.discrimination import bootstrap_ci

def cv_auroc(d, feats=FEATS, model="svm", seed=SEED):
    # 5-fold CV. One row per SUBJECT here, so folds are subject-disjoint by construction.
    X, y = d[feats].to_numpy(), d.label.astype(int).to_numpy()
    clf = (make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         SVC(probability=True, random_state=seed)) if model == "svm"
           else make_pipeline(SimpleImputer(strategy="median"),
                              RandomForestClassifier(n_estimators=400, random_state=seed, n_jobs=-1)))
    oof = np.zeros(len(y))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y):
        clf.fit(X[tr], y[tr]); oof[te] = clf.predict_proba(X[te])[:, 1]
    return bootstrap_ci(y, oof), oof

res9 = []
for tag, d in [("confounded (all)", cf), ("age-matched (honest)", am)]:
    for mdl in ["svm", "rf"]:
        (pt, lo, hi), _ = cv_auroc(d, model=mdl)
        res9.append(dict(cohort=tag, model=mdl.upper(), n_subjects=len(d),
                         AUROC=round(pt, 3), CI=f"[{lo:.3f}, {hi:.3f}]"))
print("COMPUTED HERE — classical features, 5-fold, subject-level")
pd.DataFrame(res9)

The same pattern as the embeddings: the confounded cohort looks far better than the honest
one. Age-matching costs the classical model roughly the same as it costs XLS-R — because
**both** were partly reading age.

In [ ]:
imp = make_pipeline(SimpleImputer(strategy="median"),
                    RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1))
imp.fit(am[FEATS], am.label.astype(int))
fi = pd.Series(imp[-1].feature_importances_, index=FEATS).sort_values()
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(fi.index, fi.values, color=[SPEC if i in ("jitter_local","shimmer_local","hnr","f0_mean") else WAVE for i in fi.index])
ax.set_xlabel("random-forest importance")
ax.set_title("Which classical features carry the honest signal?\n(orange = voice-quality, blue = MFCC)", fontweight="bold")
fig.tight_layout(); plt.show()
print("COMPUTED HERE — age-matched cohort")

> **Gap worth noting for the write-up.** *Monopitch* — reduced **F0 variability** — is the
> single most-cited prosodic marker of PD, and the feature set contains only `f0_mean`, not
> F0 standard deviation. On a sustained vowel that is also a direct pitch-stability measure.
> Adding it is two lines and is the most theoretically motivated extension available.
>
> Also: the MFCCs currently use librosa's music-oriented defaults (2048-point FFT, 512 hop =
> a **128 ms** window at 16 kHz) rather than the 25 ms / 10 ms speech standard. A 128 ms
> window smears across multiple glottal cycles.

---
## §10 · Self-supervised embeddings

The SSL arm replaces those 42 hand-designed numbers with 2048 learned ones, from the frozen
encoder traced in §7. Nothing about the voice is specified by us: the representation was
learned from 436,000 hours of unlabelled multilingual speech.

Let us watch the head actually learn, on real cached embeddings.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import torch.nn as nn

from pdvoice.preprocess import quality_control

# The cached embeddings are stored WITHOUT ids and aligned POSITIONALLY, so the manifest
# must be rebuilt exactly as when the cache was written: filter task, then apply QC.
nvv = nv[nv.task_kind == "vowel"].copy()
nvv, dropped = quality_control(nvv)
nvv = nvv.reset_index(drop=True)
Xnv = np.load(EMBEDDINGS / "neurovoz__xls-r-300m__vowel__L15.npz")["X"].astype(np.float32)
print(f"QC dropped {len(dropped)} recording(s) under 1.0 s; {len(nvv)} remain")
assert len(nvv) == len(Xnv), (len(nvv), len(Xnv))
ynv = nvv.label.astype(int).to_numpy()
tr, va = next(GroupShuffleSplit(1, test_size=.3, random_state=0).split(Xnv, ynv, nvv.subject_id))

torch.manual_seed(SEED)
xt, yt = torch.tensor(Xnv[tr], device=device), torch.tensor(ynv[tr], dtype=torch.float32, device=device)
xv, yv = torch.tensor(Xnv[va], device=device), torch.tensor(ynv[va], dtype=torch.float32, device=device)
head = build_head(2048).to(device)
lf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float((yt==0).sum())/float(yt.sum()), device=device))
opt = torch.optim.Adam(head.parameters(), lr=1e-3)

hist = []
for ep in range(1, 121):
    head.train(); opt.zero_grad()
    loss = lf(head(xt).squeeze(-1), yt); loss.backward(); opt.step()
    head.eval()
    with torch.no_grad():
        vl = head(xv).squeeze(-1)
        hist.append((ep, loss.item(), lf(vl, yv).item(),
                     roc_auc_score(ynv[va], torch.sigmoid(vl).cpu().numpy())))
H = pd.DataFrame(hist, columns=["epoch", "train_loss", "val_loss", "val_AUROC"])

fig, ax = plt.subplots(1, 2, figsize=(11, 2.9))
ax[0].plot(H.epoch, H.train_loss, color=WAVE, label="train"); ax[0].plot(H.epoch, H.val_loss, color=SPEC, label="val")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("BCE loss"); ax[0].legend(); ax[0].set_title("loss")
ax[1].plot(H.epoch, H.val_AUROC, color=SPEC); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("val AUROC")
ax[1].set_title("validation AUROC")
fig.suptitle("COMPUTED HERE — the 525,313-parameter head learning on frozen embeddings", fontweight="bold")
fig.tight_layout(); plt.show()
print(H.iloc[[0, 4, 19, 49, -1]].round(4).to_string(index=False))

**Train loss collapses toward zero** — 525,313 parameters can very nearly memorise a few
hundred recordings. Train loss alone therefore tells you nothing, and validation loss can get
*worse* while it improves. That is why training early-stops on validation loss and restores
the best weights.

And this is why there are **three** splits, not two:

| Split | Used for | Moves weights? |
|---|---|---|
| **train** | gradient descent | yes |
| **validation** | when to stop; fitting the calibrator; choosing the threshold | no — but it *is* spent, so cannot be reported |
| **test / external** | the reported number | no — untouched until everything is locked |

---
## Supervisor ask 4 - what is actually *in* the 2048 numbers?

This is the ask the supervisor called the core of the thesis, and it is the one that turns a
frozen encoder from a black box into something the write-up can reason about.

The method is **linear probing**: freeze the embeddings, then train a simple linear model to
predict some *known* attribute from them. If a linear model can recover an attribute, that
attribute is encoded in the representation in a directly usable form. Each probe is scored
against its own chance level, so `above_chance` is the comparable column.

The row to watch is **PD diagnosis** - the thing we actually want - read against everything
else the vector happens to know.

In [ ]:
if have("embedding_probes.json"):
    pr = pd.DataFrame(cited("embedding_probes.json"))
    pr["above_chance"] = pr.value - pr.chance
    show = (pr.sort_values("above_chance", ascending=False)
              [["attribute", "cohort", "layer", "metric", "value", "chance",
                "above_chance", "extra"]])
    display(show.round(4).reset_index(drop=True))
else:
    print("run scripts/embedding_probes.py")

In [ ]:
if have("embedding_probes.json"):
    pr = pd.DataFrame(cited("embedding_probes.json"))
    pr["above_chance"] = pr.value - pr.chance
    # One layer only, so the bars are comparable; drop the R2 row (different units).
    p = pr[(pr.layer == 15) & (pr.metric != "R2")].sort_values("above_chance")
    lbl = p.attribute + "  (" + p.cohort + ")"
    is_pd = p.attribute.str.contains("PD")
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    ax.barh(lbl, p.above_chance, color=[SPEC if x else WAVE for x in is_pd])
    ax.set_xlabel("margin above chance  (same metric, own chance level)")
    ax.set_title("What a frozen XLS-R L15 embedding encodes\n"
                 "orange = the target we actually want", fontweight="bold")
    fig.tight_layout()
    plt.show()
else:
    print("run scripts/embedding_probes.py")

**The vector knows almost everything except the disease.** Reading the table top to bottom:
**task** (vowel vs speech) is perfectly separable, **sex** is near-perfect, **corpus identity**
and **language** are far above their three-way chance level, **age** is clearly present, and
**recording device** is weak but real. **PD diagnosis sits below all of them** - and that value
is measured on the *confounded* cohort, so part of even that margin is the age signal rather
than the disease.

That ordering is the honest answer to "what do the embedding vectors mean". They are a rich
description of **who is speaking, in what language, into what device, performing which task** -
and only weakly a description of whether that person has Parkinson's. Nothing here is
surprising for a model pretrained for speech recognition: speaker and channel identity are
exactly what such a model must represent well.

Three direct consequences for the thesis:

1. **High corpus identity is why naive cross-corpus training leaks.** Pooling datasets hands the
   classifier a near-perfect dataset label, and if class balance differs between corpora it can
   score well without learning anything about voice.
2. **Near-perfect sex encoding makes the sex confound unavoidable**, not merely possible - which
   is what motivates the age+sex matching above.
3. **The PD margin is smaller than the age, sex, task, language and corpus margins.** Any claim
   built on this representation has to control for all of them, and that is the RQ3 argument in
   one table.

---
## §11 · Comparison — classical vs SSL vs fusion

The same MLP head, the same subjects, the same folds. Only the input features change: 42
classical, 2048 SSL, or 2065 fused.

In [ ]:
rows = []
if have("classical_baseline__cap1.json"):
    cb = cited("classical_baseline__cap1.json")
    for mdl in ("rf", "svm"):
        rows.append(dict(method=f"Classical {mdl.upper()}",
                         confounded=cb["scenarios"]["baseline"]["models"][mdl]["auroc"],
                         age_matched=cb["scenarios"]["age-matched"]["models"][mdl]["auroc"]))
if have("fusion__xls-r-300m__L15.json"):
    fu = cited("fusion__xls-r-300m__L15.json")
    for key, name in [("SSL-only", "SSL (XLS-R L15)"), ("FUSION", "Fusion (SSL+42)")]:
        rows.append(dict(method=name,
                         confounded=fu["scenarios"]["baseline"][key]["auroc"],
                         age_matched=fu["scenarios"]["age-matched"][key]["auroc"]))
comp = pd.DataFrame(rows).round(3)
print("LOADED classical_baseline__cap1.json + fusion__xls-r-300m__L15.json")
print("mPower vowel cap-1, subject-level AUROC\n")
display(comp)

fig, ax = plt.subplots(figsize=(7, 3))
x = np.arange(len(comp)); w = .38
ax.bar(x - w/2, comp.confounded,  w, color=WAVE, label="confounded cohort")
ax.bar(x + w/2, comp.age_matched, w, color=SPEC, label="age-matched (honest)")
ax.axhline(.5, ls=":", c="k", lw=1); ax.set_xticks(x); ax.set_xticklabels(comp.method, rotation=18, ha="right")
ax.set_ylabel("subject AUROC"); ax.set_ylim(.45, .88); ax.legend(fontsize=8)
ax.set_title("The confound inflates every method — and fusion most of all", fontweight="bold")
fig.tight_layout(); plt.show()


# ---- DeLong: are these gaps real? (paired, on the same subjects) ----------
from pdvoice.eval.discrimination import delong_roc_test

sig, ns = [], {}
for scen in ("baseline", "age-matched"):
    f = {nice: pd.read_csv(LOGS / f"fusion_preds__{scen}__{key}.csv").set_index("s")
         for key, nice in [("SSL_only", "SSL"), ("classical_only", "classical"),
                           ("FUSION", "Fusion")]}
    # join on subject_id, never on row order — the positional-join trap
    m = f["SSL"][["y"]].join([f[n]["p"].rename(n) for n in f], how="inner")
    for n in f:
        assert (m.y == f[n].y.reindex(m.index)).all(), f"label mismatch in {n}"
    ns[scen] = len(m)
    for a, b in [("Fusion", "SSL"), ("Fusion", "classical"), ("SSL", "classical")]:
        A, B, p = delong_roc_test(m.y.to_numpy(), m[a].to_numpy(), m[b].to_numpy())
        sig.append(dict(cohort=scen, comparison=f"{a} vs {b}", delta=round(A - B, 3),
                        p=round(p, 3),
                        verdict="significant" if p < .05 else "not significant"))

print(f"\nLOADED fusion_preds__*.csv — DeLong paired on the same subjects "
      f"(confounded n={ns['baseline']:,}, age-matched n={ns['age-matched']:,})\n")
display(pd.DataFrame(sig))


### Two conclusions, and the second is the interesting one

**On the confounded cohort every gap is real, and fusion is genuinely best** — fusion 0.754,
SSL 0.731, classical 0.687, all three pairwise differences significant (DeLong p < 0.001,
n = 4,809). That is the comparison most papers would report.

**On the honest age-matched signal every gap disappears.** Same test, same 1,124 subjects:

| Comparison | ΔAUROC | p | Verdict |
|---|---|---|---|
| Fusion vs SSL | −0.013 | 0.338 | not significant |
| Fusion vs classical | +0.013 | 0.492 | not significant |
| SSL vs classical | +0.026 | 0.222 | not significant |

Absolute values: **SSL 0.607, fusion 0.594, classical 0.580.**

**All three are statistically indistinguishable — and fusion is not the winner.** It sits
*below* SSL alone. Once age is controlled, 42 interpretable coefficients are competitive
with a 300-million-parameter multilingual transformer *on the sustained-vowel task*. Most of
SSL's apparent advantage was confound exploitation.

The **direction** of the fusion result is the point: fusion helps only where the confound
is. Adding the classical features to SSL buys a significant +0.023 on the confounded cohort
and nothing at all (−0.013, n.s.) once age is matched. Combining the two feature sets did
not add signal — it added a second route to the same nuisance variable.

§7 gives a principled reason the honest tie is not surprising: XLS-R tokenises at 20 ms, so
it cannot measure jitter as a period perturbation at all — only its spectral consequence.
The Praat features measure it directly. The two are partly measuring **different physics**,
so neither strictly dominates.

**Where SSL is genuinely irreplaceable:** connected speech (0.86–0.95 within, where the
classical features are not valid) and multilingual transfer. Fusion is *not* a good default
on the honest signal: it costs 2,048 extra dimensions to land below SSL alone. If
interpretability is what you want, take the 42-feature classical model on its own — 0.580
against SSL's 0.607, a gap this test cannot separate from zero.

> **This reverses an earlier version of this section**. The previous table reported
> fusion *ahead* of SSL by +0.023 on the honest signal. That came from a bespoke code path
> in `fusion.py` that joined the classical features to the embeddings **by row position**
> and returned an SSL number (0.671) disagreeing with every other measurement of the same
> quantity. Rebuilt on the shared cohort path with a join on `subject_id`, the sign flips.
> The *conclusion* — all three indistinguishable — survives; the ranking does not.


### Does the choice of classifier change any of this?

The field does not use an MLP head: Klempíř 2024 uses Random Forest and XGBoost, VoiceFM
uses **linear probes** on frozen embeddings, and cross-lingual PD work uses SVM. Linear
probing is also the SUPERB convention for frozen representations, because it measures what
is *in* the embedding rather than what a nonlinear head can learn from it.

Our head has 525,313 parameters for roughly 1,100 subjects, so it can memorise — we watched
it do so in §10. So it is worth checking whether any conclusion depends on that choice.

In [ ]:
if have("classifier_comparison.csv"):
    cc = pd.read_csv(LOGS / "classifier_comparison.csv")
    sw = cc[cc.features != "age only (1)"]
    spread = (sw.groupby(["cohort", "features"])
                .AUROC.agg(["min", "max"])
                .assign(range=lambda d: (d["max"] - d["min"]).round(3)).round(3))
    print("spread across logreg / SVM / RandomForest / HistGradBoost / MLP head:\n")
    display(spread)
else:
    print("run scripts/classifier_comparison.py")

The spread across five classifiers is **smaller than the gap between feature sets**, and far
smaller than the confound effect. The conclusions are therefore not an artefact of the head:
SVM and Random Forest in fact score slightly **above** our MLP on age-matched embeddings, and
the linear probe slightly below — so there is genuine nonlinear structure, but only a little
of it.

This also retires a real methodological problem: the original comparison scored classical
features with SVM/RF but SSL embeddings with the MLP, changing **two variables at once**.
Here every classifier sees every feature set on identical folds.

---
## Supervisor ask 3 - an explicit objective function

Every number so far has been AUROC, which ranks models but encodes **no clinical preference**:
it treats a missed patient and a false alarm as equally interesting, and it never has to say
what the model would be *used for*.

The ask was to state an objective. This one is stated as a **threshold probability**
`p_t = 0.10`, which is a formal commitment: *we would accept 9 false positives to catch 1 true
case*. That is defensible for a screening tool whose false positives cost a clinic visit and
whose false negatives cost years of untreated progression.

From `p_t` follows **net benefit** - decision curve analysis:

```
NB = TP/n - (FP/n) * pt/(1-pt)
```

and the benchmark that matters is not another model. It is **"screen everyone"**: refer every
person who walks in. A model earns its place only by beating that.

The cohort is 50% PD by construction, so net benefit is also reported at a realistic
population prevalence - the `*_at_target_prev` columns.

In [ ]:
if have("objective_function.json"):
    ob = cited("objective_function.json")
    print(f"threshold probability pt = {ob['pt']}  "
          f"=> willing to accept {ob['fp_per_tp_accepted']:.0f} FP per TP")
    print(f"specificity floor for the constrained objective: {ob['spec_floor']}")

    r = pd.DataFrame(ob["rows"])
    show = (r[r.cohort == "age-matched"]
            .sort_values("NB_at_target_prev", ascending=False)
            [["features", "classifier", "AUROC", "sens", "spec",
              "NB_at_target_prev", "NB_all_at_target_prev", "beats_screen_all",
              "sens@spec>=0.7"]])
    print("\nAGE-MATCHED cohort, ranked by net benefit at the target prevalence")
    display(show.round(4).reset_index(drop=True))
else:
    print("run scripts/objective_function.py")

In [ ]:
if have("objective_function.json"):
    ob = cited("objective_function.json")
    r = pd.DataFrame(ob["rows"])
    am = r[r.cohort == "age-matched"].copy()

    # Does the objective pick a different winner than AUROC does?
    by_auroc = am.sort_values("AUROC", ascending=False).iloc[0]
    by_nb = am.sort_values("NB_at_target_prev", ascending=False).iloc[0]
    by_sens = am.sort_values("sens@spec>=0.7", ascending=False).iloc[0]
    pick = pd.DataFrame([
        dict(criterion="max AUROC", features=by_auroc.features,
             classifier=by_auroc.classifier, value=round(by_auroc.AUROC, 4)),
        dict(criterion=f"max net benefit @ pt={ob['pt']}", features=by_nb.features,
             classifier=by_nb.classifier, value=round(by_nb.NB_at_target_prev, 5)),
        dict(criterion="max sens @ spec>=0.7", features=by_sens.features,
             classifier=by_sens.classifier,
             value=round(by_sens["sens@spec>=0.7"], 4)),
    ])
    print("Does the objective function change which model you would ship?")
    display(pick)

    # The number that actually matters: how much better than screening everyone?
    gain = (am.NB_at_target_prev - am.NB_all_at_target_prev).max()
    print(f"best net-benefit gain over 'screen everyone': {gain:.5f}")
    print(f"  = {gain * 1000:.1f} extra true cases per 1,000 people screened, "
          "at no extra false-positive cost")
    print(f"models beating 'screen everyone' at all: "
          f"{int(am.beats_screen_all.sum())}/{len(am)}")
else:
    print("run scripts/objective_function.py")

**The objective function does not change the ranking - and that is itself the finding.** The
model AUROC would pick is also the model net benefit picks, so the AUROC-based reporting in
this notebook is not misleading about *ordering*. That was previously an assumption; it is now
a measurement.

**But it changes the conclusion completely.** Look at the margin over "screen everyone": the
best model buys on the order of **one extra true case per 1,000 people screened**. For
comparison, the age-only model on the confounded cohort buys tens of cases per thousand. A
screening tool that adds one case per thousand over referring everybody is not a clinically
useful instrument, and no AUROC in this notebook says that out loud.

Notice too why the raw `sens`/`spec` columns look strange - sensitivity near 1.00 with
specificity near 0.00. At `p_t = 0.10` in a 50%-prevalence cohort, the rational action really
is to flag almost everyone; that is the honest behaviour of the stated objective, not a bug.
The `sens@spec>=0.7` column is the more legible operating point, and it re-ranks the models
differently from net benefit - which is exactly why the objective has to be declared in advance
rather than chosen after seeing the results.

---
# PART IV — CROSS-LANGUAGE

## §12 · Spanish — NeuroVoz

Now the RQ1 test. The model is trained on **English** mPower sustained vowels, then
**locked** — head weights, calibration map and threshold all frozen — and applied to
**Spanish** NeuroVoz without any retraining.

In [ ]:
# Both lines come from ONE script, the same five seeds and the same layers: the only
# thing that changes between them is who is in the cohort. The earlier version of this
# cell drew the second line from a different script and a different cap, which
# confounded the comparison with the calibration path.
LS = "label_sensitivity.json"
COLS = ["within", "NeuroVoz_es", "IPVS_it", "MDVR_en"]
KEYS = ["within", "transfer_neurovoz", "transfer_ipvs", "transfer_mdvr_kcl"]
LBL = {"baseline": "Confounded mPower", "agematched": "Age-matched, same script"}
mean_ = None
if have(LS):
    ls = cited(LS)
    mean_ = pd.DataFrame({LBL[s]: np.array([[r[k] for k in KEYS] for r in ls["runs"][s]],
                                           float).mean(0) for s in LBL}, index=COLS).T
    sd_ = pd.DataFrame({LBL[s]: np.array([[r[k] for k in KEYS] for r in ls["runs"][s]],
                                         float).std(0, ddof=1) for s in LBL}, index=COLS).T
    print(f"LOADED {LS} - {len(ls['seeds'])} seeds, cap {ls['cap']}, "
          f"within layer {ls['within_layer']} / transfer layer {ls['transfer_layer']}")
    display(mean_.round(3))

# The headline honest model is a stricter setting: cap-10, and head, calibrator AND
# threshold are all frozen before any external data is touched.
SCALE = "scaleup_agematched__xls-r-300m__L15__cap10.json"
if have(SCALE):
    sc = cited(SCALE)
    def ms(get):
        v = np.array([get(r) for r in sc["runs"]], float)
        return f"{v.mean():.3f}+/-{v.std(ddof=1):.3f}"
    print(f"LOADED {SCALE} - {len(sc['seeds'])} seeds, cap {sc['cap']}, "
          f"n={sc['runs'][0]['n_subjects']} subjects, locked head+calibrator+threshold")
    display(pd.DataFrame([dict(training="Age-matched, SCALED + LOCKED",
                               within=ms(lambda r: r["within"]["auroc"]),
                               NeuroVoz_es=ms(lambda r: r["transfer"]["neurovoz"]["auroc"]),
                               IPVS_it=ms(lambda r: r["transfer"]["ipvs"]["auroc"]),
                               MDVR_en=ms(lambda r: r["transfer"]["mdvr_kcl"]["auroc"]))]))

if mean_ is not None:
    fig, ax = plt.subplots(figsize=(7.5, 3))
    for (name, row), c, m in zip(mean_.iterrows(), [WAVE, SPEC], ["o", "s"]):
        ax.errorbar(COLS, row.to_numpy(), yerr=sd_.loc[name].to_numpy(), marker=m,
                    color=c, label=name, lw=2, capsize=3)
    ax.axhline(.5, ls=":", c="k", lw=1); ax.set_ylabel("subject AUROC")
    ax.set_ylim(.3, .87); ax.legend(fontsize=8)
    ax.set_title("Removing the confound: within falls, transfer rises", fontweight="bold")
    fig.tight_layout(); plt.show()


### Read the two lines against each other — this is the RQ1/RQ2 answer

The **confounded** model starts high (**0.801 ± 0.050**) and falls apart: Spanish drops to
**0.607 ± 0.023**, English-MDVR to **0.431 ± 0.087**, and Italian to **0.404 ± 0.106** —
below chance on average. That looks like a catastrophic failure of generalisation.

The **age-matched** model starts far lower (**0.617 ± 0.028**) and then barely moves: Italian
**0.624 ± 0.086**, Spanish **0.601 ± 0.054**, English-MDVR **0.590 ± 0.123**. Same script,
same five seeds, same layers — the only change is who is in the cohort. Within falls **0.185**
while mean transfer rises **0.124**, and it is the *pair* of movements that identifies a
confound: better modelling would raise both, and label noise (tested the same way, on the same
seeds) moves neither.

**Why Italian inverts.** The likely reading is that a model trained on mPower has largely
learned *older ⇒ PD*, and IPVS gives it nothing to grip on: 22 of its 37 controls are in the
elderly group, against 28 patients. Note this is the plausible mechanism, not a verified one
— the IPVS release carries a young/elderly group label but no numeric ages, so the ranking
cannot be checked directly. Treat it as an interpretation.

**The scaled, locked model is more equivocal, and that difference matters.** At cap-10, with
head, calibrator *and* threshold frozen before any external data is touched, within is
**0.649 ± 0.018** and the externals are **0.559 ± 0.044** (es), **0.621 ± 0.103** (en),
**0.552 ± 0.119** (it). The Spanish drop shrinks from 0.19 to 0.09 — real, and the clearest
case. The other two carry SDs of ±0.10–0.12, wide enough to neither confirm nor refute the
gap closing. So the defensible claim is **"the gap closes for Spanish, and the other two are
inconclusive at this sample size"** — not "the transfer gap vanishes", which an earlier draft
of this section asserted on a single seed.

So the "generalisation crisis" is largely a confound, not a property of voice models. The
honest signal is **modest (~0.62–0.65) and, in Spanish at least, portable**. A flat, low line
is scientifically far more valuable than a high line that collapses — but the flatness is
established for one language, not three, and the fix for that is the next section, not a
bigger model.

### But Spanish also has connected speech — and that changes the picture entirely


In [ ]:
DS = ["neurovoz", "ipvs", "mdvr_kcl"]
mtx = pd.DataFrame(np.nan, index=[f"train {d}" for d in DS], columns=[f"test {d}" for d in DS])
for src in DS:
    f = f"transferany__xls-r-300m__train-{src}-speech__L15.json"
    if not have(f):
        print(f"  MISSING {f} -> run scripts/transfer_any.py --train {src}:speech "
              f"--tests ... --layer 15")
        continue
    t = cited(f)
    mtx.loc[f"train {src}", f"test {src}"] = round(t["within"]["auroc"], 3)
    for tgt, v in t["transfer"].items():
        mtx.loc[f"train {src}", f"test {tgt.split(':')[0]}"] = round(v["auroc"], 3)
print("LOADED transferany__*.json · CONNECTED SPEECH, XLS-R L15")
print("diagonal = within 5-fold, off-diagonal = locked transfer\n")
display(mtx.style.format("{:.3f}", na_rep="-").background_gradient(cmap="YlGn", vmin=.5, vmax=1))

off = [mtx.iloc[i, j] for i in range(3) for j in range(3) if i != j]
print(f"\nCOMPUTED HERE from the matrix: off-diagonal min {min(off):.3f}, "
      f"max {max(off):.3f}, mean {np.mean(off):.3f}")

**Every off-diagonal speech transfer is 0.70–0.85, mean ~0.77** — in all directions, across
Spanish, Italian and English. Compare the vowel transfers of 0.13–0.62. Train on Spanish
speech and you get 0.95 within, 0.84 to English, 0.70 to Italian.

**So the task matters more than the model, the language, or the confound.** That is §2's
four-systems table showing up in the numbers: a sustained vowel exposes phonation only, while
connected speech also carries prosody, articulation and fluency — and those deficits are
motor, so they survive a change of language.

Verification that these high within-dataset scores are genuine rather than artifacts:

| Check | NeuroVoz speech (0.95) | IPVS speech (1.00) |
|---|---|---|
| Label permutation | 0.494 — no CV leak | 0.459 — no CV leak |
| Age-matched | 0.948 — not age | — |
| Sex-stratified | 0.907 / 0.914 — not sex | — |
| Recording dates | overlap — not a batch split | — |
| Drop young controls | — | 0.981 |
| Drop the 44.1 kHz PD batch | — | 1.000 (§6) |

Both survive. The IPVS 1.00 remains **optimistic** for a different reason — 65 speakers, one
site, anechoic scripted reading, near-ceiling separability — which is why the 0.70–0.85
transfer figures, not the diagonal, are the honest estimate.

**The strategic implication:** mPower being vowel-only is the ceiling on this project. The
highest-value next dataset is a **clinician-labelled connected-speech corpus** (Bridge2AI).

### What does 0.62 actually mean for a patient? — the metrics AUROC hides

Every result in this notebook has been reported as AUROC, because that is what the field
reports. AUROC is a *ranking* measure: it asks whether a randomly chosen patient scores above
a randomly chosen control. It never has to commit to a threshold, so it never has to say how
many patients the model would actually find.

Each evaluation stores much more than that, and the rest of it is far less flattering. Three
families:

- **discrimination** — AUROC, and AUPRC, which is the honest one when a decision is made at a
  single operating point
- **calibration** — Brier, ECE, MCE: are the predicted probabilities *numerically* right, or
  merely ordered right?
- **an operating point** — sensitivity at a fixed specificity, which is the number a clinician
  would actually ask for

In [ ]:
import glob

tm_rows = []
for f in sorted(glob.glob(str(LOGS / "layer2__xls-r-300m__*__L15.json"))):
    d = json.loads(Path(f).read_text())
    t = d.get("at_locked_threshold", {})
    s90 = d.get("sensitivity_at_90_spec", {})
    n_pd, n_hc = d.get("n_pd"), d.get("n_hc")
    tm_rows.append(dict(
        target=d.get("dataset"), n=d.get("n_subjects"),
        # AUPRC has no fixed chance level: its baseline IS the prevalence, so an AUPRC
        # must be read against this column or it means nothing.
        prevalence=n_pd / (n_pd + n_hc) if n_pd and n_hc else np.nan,
        AUROC=d["auroc"], AUPRC=d["auprc"],
        ECE=d["ece"], MCE=d.get("mce"), Brier=d["brier"],
        sens=t.get("sensitivity"), spec=t.get("specificity"), F1=t.get("f1"),
        **{"sens@90%spec": s90.get("sensitivity")}))

if tm_rows:
    tm = pd.DataFrame(tm_rows).set_index("target")
    print("A LOCKED mPOWER MODEL, EVALUATED ABROAD — every stored metric")
    display(tm.round(3))
    lift = (tm.AUPRC - tm.prevalence).round(3)
    print("AUPRC minus prevalence (how much better than guessing at one threshold):")
    print(lift.to_string())
else:
    print("run scripts/layer2_transfer.py")

In [ ]:
if tm_rows:
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    x = np.arange(len(tm)); w = 0.38
    ax.bar(x - w / 2, tm.AUROC, w, color=WAVE, label="AUROC (what we report)")
    ax.bar(x + w / 2, tm["sens@90%spec"], w, color=SPEC,
           label="sensitivity @ 90% specificity")
    ax.axhline(.5, ls=":", c="k", lw=1)
    ax.set_xticks(x); ax.set_xticklabels(tm.index)
    ax.set_ylabel("score"); ax.set_ylim(0, 1); ax.legend(fontsize=8)
    ax.set_title("The same three models, two ways of asking how good they are",
                 fontweight="bold")
    fig.tight_layout(); plt.show()

**AUROC is the most flattering number available**, and the gap is not small. NeuroVoz reads as
0.618 — mediocre but alive. The same model at a usable operating point finds about **15% of
patients**, and IPVS and MDVR-KCL are worse still at 14% and 13%.

**Read AUPRC against the prevalence column, never against 0.5.** AUPRC has no fixed chance
level — its baseline *is* the prevalence of the target. On IPVS the model scores 0.525 where
guessing scores 0.560, so at a single threshold it is **worse than useless**; the AUROC of
0.356 was already telling us the ranking is inverted there. NeuroVoz is the only target with a
meaningful margin (+0.136).

**Calibration fails harder than discrimination.** ECE runs 0.22–0.40 off-domain against ~0.03
in-domain. The model is not merely less accurate abroad — the probabilities it emits are
numerically wrong, so any downstream decision rule calibrated at home silently mis-fires. A
discrimination-only report hides this completely, which is why locked cross-dataset evaluation
*with calibration* is requirement 3 of the five.

The `sens` and `spec` columns are at the **locked** threshold — chosen on mPower validation
data and then frozen, never re-tuned on the target. Re-tuning it abroad would be a different
and much easier experiment, and it is the one the literature usually runs.

---
## §12b · How do these numbers compare with the published literature?

The natural worry about everything above is that the numbers look low. They do — against
*headline* figures. This section checks that directly rather than asserting it.

The published side of the table is typed by hand, because it comes from papers rather than
from our pipeline; it lives in `references/published_results.csv` with a citation per row.
**Our side is not typed** — `scripts/literature_comparison.py` reads it from the same
artifacts as the rest of this notebook, so a re-run cannot leave the comparison stale.

The `comparability` column is the point of the exercise. A published number is a benchmark
only if it was measured the same way, and one row here was not.

In [ ]:
if have("literature_comparison.json"):
    lc = cited("literature_comparison.json")
    L = pd.DataFrame(lc["rows"])
    show = L[["claim", "task", "published", "published_sd", "ours", "delta",
              "verdict", "comparability", "source"]]
    display(show.round(3))
    print(f"comparable claims : {lc['n_comparable']} of {lc['n_claims']}")
    print(f"mean |difference| : {lc['mean_abs_delta']:.3f} AUROC")
else:
    print("run scripts/literature_comparison.py")

In [ ]:
if have("literature_comparison.json"):
    L = pd.DataFrame(cited("literature_comparison.json")["rows"]).dropna(subset=["ours"])
    ok = L[L.comparability != "NOT comparable"]
    bad = L[L.comparability == "NOT comparable"]

    fig, ax = plt.subplots(figsize=(8, 3.6))
    y = np.arange(len(L))
    ax.scatter(L.published, y, s=46, color=SPEC, label="published", zorder=3)
    ax.scatter(L.ours, y, s=46, color=WAVE, marker="D", label="ours", zorder=3)
    for i, r in enumerate(L.itertuples()):
        ax.plot([r.published, r.ours], [i, i], color="k", lw=1, alpha=.35, zorder=2)
    for i in bad.index:
        ax.axhspan(i - .45, i + .45, color="crimson", alpha=.10, zorder=1)
    ax.set_yticks(y)
    ax.set_yticklabels([c[:46] for c in L.claim], fontsize=7.5)
    ax.set_xlabel("AUROC"); ax.set_xlim(0.4, 1.05); ax.legend(fontsize=8, loc="lower left")
    ax.set_title("Ours against the literature, claim by claim\n"
                 "shaded row = not measured the same way, so not a benchmark",
                 fontweight="bold")
    fig.tight_layout(); plt.show()
    print(f"mean |difference| over the {len(ok)} comparable claims: "
          f"{ok.delta.abs().mean():.3f} AUROC")

**The results are concordant with the field, not anomalous.** Across the seven comparable
claims the mean absolute difference is about **0.04 AUROC**, and the largest is 0.07. Both
directions occur — some of ours are higher, some lower — which is what agreement looks like;
a systematic offset in one direction would suggest a pipeline error instead.

Two rows deserve their flags rather than a tick.

**The 0.94 mPower row is in the table to be criticised, not matched.** That study splits at the
**recording** level, so with one mPower speaker contributing 355 recordings the same person
appears in training and test. It also *restricts* age to 50–70 rather than matching it, leaving
a 3.5-year gap, and leaves an 18-point sex gap uncontrolled. Comparing our 0.80 to their 0.94
would be comparing a subject-level result to a recording-level one. This resolves what looks
like a contradiction with our own finding: their number is inflated by leakage, not earned by
better confound control.

**The NeuroVoz vowel row carries a caveat that applies to both sides.** NeuroVoz vowel task
codes are unevenly distributed between the groups — the third repetition of each vowel is
about 90% patients — so a speaker's task codes alone predict the diagnosis at AUROC 0.895 with
no audio. Neither our 0.610 nor the published 0.646 is confound-free, and the agreement between
them should not be read as mutual validation.

**And the IPVS ceiling is a ceiling, not a win.** Both figures sit at ~1.00 on 50 speakers, in
a corpus whose 44.1 kHz batch is PD-only. Reproducing a ceiling demonstrates that the pipeline
works; it demonstrates nothing about Parkinson's.

The honest summary for the write-up: **where a comparison is fair, our results match the
literature; where they differ, the difference is explained by evaluation protocol rather than
by modelling.** That is the thesis argument, stated as a measurement.

---
## Key numbers, recomputed from the artifacts

In [ ]:
# Everything the discussion below refers to, read back from disk at run time.
# If a cohort or a script changes, THESE numbers change and the prose must follow.
print("CONFOUND DIAGNOSIS  (label_sensitivity.json, 5 seeds)")
if have("label_sensitivity.json"):
    ls = cited("label_sensitivity.json")
    TR = ["transfer_ipvs", "transfer_neurovoz", "transfer_mdvr_kcl"]
    base = None
    for sc, runs in ls["runs"].items():
        w = np.array([r["within"] for r in runs], float)
        t = {k: np.array([r[k] for r in runs], float) for k in TR}
        if base is None:
            base = (w.mean(), {k: v.mean() for k, v in t.items()})
        d_tr = np.mean([t[k].mean() - base[1][k] for k in TR])
        cells = "  ".join(f"{k[9:]} {t[k].mean():.3f}+/-{t[k].std(ddof=1):.3f}" for k in TR)
        print(f"   {sc:<11} within {w.mean():.3f}+/-{w.std(ddof=1):.3f}   {cells}"
              f"   [vs baseline: within {w.mean() - base[0]:+.3f}, transfer {d_tr:+.3f}]")
    print(f"   mPower age gap: PD {ls['age_pd']} vs HC {ls['age_hc']}"
          f"  ({ls.get('age_gap'):.1f} yr, per {ls.get('age_level', '?')})")

print("\nHONEST MODEL, SCALED  (scaleup_agematched cap10)")
SC = "scaleup_agematched__xls-r-300m__L15__cap10.json"
if have(SC):
    sc = cited(SC)
    for key, lbl in [(None, "within mPower"), ("neurovoz", "-> NeuroVoz (es)"),
                     ("ipvs", "-> IPVS (it)"), ("mdvr_kcl", "-> MDVR-KCL (en)")]:
        v = np.array([(r["within"] if key is None else r["transfer"][key])["auroc"]
                      for r in sc["runs"]], float)
        print(f"   {lbl:<18} {v.mean():.3f} +/- {v.std(ddof=1):.3f}   "
              f"(seeds {sc['seeds']}, n={sc['runs'][0]['n_subjects']} subjects)")

print("\nDEMOGRAPHICS-ONLY BASELINE  (classifier_comparison.csv)")
if have("classifier_comparison.csv"):
    cc = pd.read_csv(LOGS / "classifier_comparison.csv")
    for coh in ("confounded", "age-matched"):
        b = cc[cc.cohort == coh].groupby("features").AUROC.max()
        print(f"   {coh:<12} " + "   ".join(f"{k}: {v:.3f}" for k, v in b.items()))

print("\nIPVS BATCH TEST  (verify_ipvs_batch.json)")
if have("verify_ipvs_batch.json"):
    b = cited("verify_ipvs_batch.json")
    print(f"   rate alone {b['rate_alone_auroc']:.3f} | baseline {b['baseline']['auroc']:.3f}"
          f" | drop 44.1k PD {b['drop_44k_pd']['auroc']:.3f}"
          f" | + drop young HC {b['drop_44k_pd_and_young_hc']['auroc']:.3f}")

print("\nSPEECH TRANSFER  (transferany)")
for src in ["neurovoz", "ipvs", "mdvr_kcl"]:
    f = f"transferany__xls-r-300m__train-{src}-speech__L15.json"
    if have(f):
        t = cited(f)
        off = "  ".join(f"{k.split(':')[0]} {v['auroc']:.3f}" for k, v in t["transfer"].items())
        print(f"   train {src:<10} within {t['within']['auroc']:.3f}   ->  {off}")

> The discussion that follows restates the block above. **Those printed values are the
> authoritative ones** - if a script is re-run on a different cohort, trust the output,
> not the prose.

---
# PART V — ANSWERS TO THE RESEARCH QUESTIONS

## RQ1 — How well do current single-modality voice methods generalise across datasets and languages?

**On the sustained-vowel task, badly — but mostly for a reason the literature misdiagnoses.**

A conventionally-trained mPower model scores **0.801 ± 0.050** within-dataset (5 seeds) and
drops to **0.607 ± 0.023** on Spanish (NeuroVoz), **0.431 ± 0.087** on English at a different
task (MDVR-KCL), and **0.404 ± 0.106** on Italian (IPVS) — below chance *on average*, though
individual seeds run 0.31 to 0.70, so no single draw from that spread is worth quoting. This
reproduces the field's pattern: in-dataset 95–99% falling to 70–85% on independent data.

But the collapse is **largely an age confound, not a failure of voice modelling**. mPower
controls are ~27 years younger than its patients (subject-level HC 32.6 vs PD 60.0). The embeddings do
encode age — though **not as strongly as an early pilot suggested**: on mPower controls the
young-vs-elderly probe gives AUROC **0.761**, and continuous age **R² 0.177**. The AUROC 1.0
once quoted here was IPVS-specific, a small anechoic corpus with an extreme age contrast, and
does not generalise.

The diagnosis is the *asymmetry* between two interventions, each over 5 seeds. Cleaning the
labels moves the within score **+0.011** and mean transfer **+0.010** — nothing. Age-matching
moves within **−0.185** and mean transfer **+0.124**, turning Italian from 0.404 ± 0.106 into
0.624 ± 0.086. Destroying apparent within-dataset performance *while improving genuine
transfer* is the signature of a confound; label noise shows neither effect.

Two further RQ1 findings. **Task shift costs more than language shift** — English MDVR-KCL
0.431 ± 0.087 against Spanish NeuroVoz 0.607 ± 0.023 — so the fragility is not specifically
linguistic. (An earlier single-seed draft read these as comparable, 0.601 vs 0.618; the
multi-seed re-run separates them, and the direction now agrees with the connected-speech
result below.) And **calibration breaks off-domain far worse than discrimination** (ECE 0.03
internal → 0.16–0.36 external), which discrimination-only reporting hides entirely.

## RQ2 — How can a method be designed to generalise while keeping discrimination comparable?

**The model is not the lever.** Four independent comparisons say so: classical features vs
SSL, five classifiers, five encoders, and fusion are *all*
statistically indistinguishable on the honest signal. What moves the number is the study
design and the task.

| Lever | Result |
|---|---|
| Statistical domain adaptation (HC-centroid / standardise) | ❌ **negative** — IPVS −0.20, NeuroVoz +0.00, MDVR +0.05. The gap resists first/second-order correction, and **why it fails is not established**: `centroid_confound_test.py` falsified both candidate explanations. |
| Mid-layer transfer (L15 vs last) | ⚠️ **partial** — it removes the sub-chance inversion on IPVS. But **L15 was selected on the external test sets**, which biases every transfer number in this notebook upward (V2); and on a fair equal-budget sweep the last layer is the *worst* layer for all five encoders, so this is a general property, not an XLS-R finding. |
| **Control the confound (age-matching)** | ✅ **works — but only reliably for Spanish.** Within 0.649 ± 0.018; externals 0.559 ± 0.044 (es), 0.621 ± 0.103 (en), 0.552 ± 0.119 (it). The Spanish gap narrows from ~0.19 to ~0.09; the other two carry SDs too wide to support a claim either way. Calibration improves but is not fixed (ECE 0.066 internal → 0.145–0.317 external). |
| **Use connected speech, not vowels** | ✅ **works best** — 0.86–0.95 within, **every** cross-language transfer 0.70–0.85, in all directions across 3 languages. |

So the answer to "keeping discrimination comparable to within-dataset performance" is
uncomfortable but honest: on vowels you achieve comparability by *lowering* the within-dataset
number to its unconfounded value (**~0.65**), not by raising the external one — and even then
only Spanish is demonstrably repaired. On connected speech you get genuine comparability at a
*useful* level (~0.78 transfer against 0.86–0.95 within). The corollary is worth stating
plainly: a **high** within-dataset score predicts a **worse** transfer.

## RQ3 — How should evaluation be designed to judge trustworthiness, and which measures support consistent comparison?

**Four questions, not one number.** Most papers report AUROC and stop.

| Question | Measures | Why it matters |
|---|---|---|
| Can it tell them apart? | AUROC, AUPRC, 95% bootstrap CI | baseline — but ranking alone never justifies a decision |
| Is the difference real? | DeLong, paired bootstrap, McNemar | stops claiming wins inside the noise — e.g. fusion vs SSL, Δ −0.013, p = 0.338 |
| Do the probabilities mean anything? | Brier, ECE, MCE, reliability curve | a model can rank perfectly and still be uninterpretable to a clinician; ours degrades 0.066 → 0.15–0.32 off-domain |
| Would using it help? | Decision Curve Analysis at a stated threshold | net benefit against screen-everyone / screen-no-one. At pt = 0.10, re-expressed at the natural prevalence 0.187, the honest model gains **+1.3 true cases per 1,000 screened** — against **+60.5** for age alone with no microphone |

Six design requirements this project argues for, each earned from a specific failure found here:

1. **Subject-disjoint splits everywhere** (`StratifiedGroupKFold`), and evaluate at **subject** level. One mPower speaker has 355 recordings and another has 1.
2. **Report demographics of both groups, and age-match or covariate-adjust.** This is the single highest-impact requirement — a ~27-year age gap silently produced most of an apparent 0.80, and on that same confounded cohort **age alone, with no audio at all, scores 0.922** against SSL's 0.796.
3. **Locked cross-dataset evaluation** — freeze weights, calibrator *and* threshold before touching external data. Report calibration off-domain, not just discrimination.
4. **Verify suspiciously high scores, with tests that can actually fail.** Permutation testing detects plumbing leaks but is **blind to class-correlated confounds**, because it destroys real and artifact signal alike. Finding the IPVS batch needed an explicit leave-the-batch-out test.
5. **Report multiple seeds.** A single run of the IPVS transfer can land anywhere from 0.31 to 0.70 (SD 0.166). A single-seed pilot cannot distinguish a finding from a draw.
6. **State the task and the preprocessing recipe precisely.** "Vowel" meant `/a/` in one corpus and five vowels in another; "speech" ranged from a 1.75 s word to 141 s of dialogue. And a mel spectrogram is undefined without naming the scale and the power/amplitude convention.


---
## §13 · Limitations

**All numbers are pilot-scale.** Capped recordings per subject (cap-1 or cap-3 of up to 355),
a single seed, and the age-matched design reduces mPower to 1,118 subjects. Only a fraction of
the 57,612 usable recordings is used. These will shift when scaled.

**Small external sets.** MDVR-KCL n=37, IPVS n=65 → wide confidence intervals. The speech
story rests partly on small clinical corpora, and IPVS within-speech 1.00 reflects a small,
clean, single-site dataset.

**mPower labels are self-report** (`professional-diagnosis`), not clinician-confirmed. Label
sensitivity analysis suggests this is not the bottleneck, but it remains a validity threat.

**Two age-match implementations differ slightly** — `label_sensitivity.py` uses an age ≥ 50
cutoff (within 0.679); `agematched_eval.py` uses proper 5-year-bin matching (within 0.621).
The latter is canonical.

**Task definitions are still uncontrolled across corpora** (§2), so vowel transfer confounds
language with phoneme.

**Known open methodological items:** MFCCs use librosa's 128 ms music-oriented window rather
than the 25 ms speech standard; F0 standard deviation (monopitch) is missing from the feature
set; quality control never opens the audio; the head trains full-batch with a 200-epoch cap,
which should be checked for convergence before scaling.

**No clinician-labelled English speech corpus yet** (Bridge2AI pending), so the cleanest
confirmation of the confound diagnosis is not yet possible.

**Layer 3** compares against *published* numbers rather than head-to-head reimplementations.

---

### Sources for every number above

```
COMPUTED HERE   inventory · task codes · cohort flow · age distributions · age-matching ·
                classical features + CV · head training curve · IPVS rate composition
CITED           outputs/logs/layer1_mpower__*.json      encoder & layer sweep
                outputs/logs/agematched__*.json         the honest locked model
                outputs/logs/classical_baseline__*.json classical vs SSL
                outputs/logs/fusion__*.json             fusion + DeLong tests
                outputs/logs/layer2__*.json             locked transfer
                outputs/logs/transferany__*.json        speech transfer matrix
                outputs/logs/verify_ipvs_batch.json     IPVS batch test
```